# 2. Behaviour Table — Merge MEG epochs with behavioural data + PROBE model

## Pipeline Step 2: Build master metadata with PROBE regressors

**Inputs (from previous steps):**
- `{behav_dir}/Mnemosyne_*_run*.mat` — raw behavioural data
- `{derivatives}/epochs_combined_{C|S}.fif` — MEG epochs from Step 1

**Outputs:**
- `{behav_dir}/master_behavior.csv` — all behavioural events
- `{behav_dir}/master_behavior_fb_unique.csv` — feedback events only
- `{behav_dir}/master_behavior_with_probe.csv` — **with PROBE regressors** ★
- `epochs_with_run_{C|S}-epo.fif` — epochs with run/trial labels
- `{derivatives}/epochs_with_master_{C|S}-epo.fif` — **key file for GLM** ★

---

## Only change `SUBJECT` in the first cell!

All other cells pick up their paths from config automatically.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.io as sio
import mne
import re


SUBJECT = "16_sub"   # name of the folder in which subject's data


MEG_ROOT = Path(r"E:\Develop\MEG")

SUBJECT_MAPPING = {
    "1_subj":  MEG_ROOT / "1_subj",
    "6_subj":  MEG_ROOT / "6_subj",
    "8_subj":  MEG_ROOT / "8_subj",
    "9_subj":  MEG_ROOT / "9_subj",
    "12_subj": MEG_ROOT / "12_subj",
    "2_subj":  MEG_ROOT / "2_subj",
    "11_subj": MEG_ROOT / "11_subj",
    "7_subj":  MEG_ROOT / "7_subj",
    "10_subj": MEG_ROOT / "10_subj",
    "15_subj": MEG_ROOT / "15_subj",
    "16_subj": MEG_ROOT / "16_subj",
}

if SUBJECT not in SUBJECT_MAPPING:
    raise ValueError(f"❌ Subject '{SUBJECT}' not found!\n"
                     f"   Available: {list(SUBJECT_MAPPING.keys())}")

BASE = SUBJECT_MAPPING[SUBJECT]
BEHAV_DIR = BASE / "behav"

# C
DERIV_C = BASE / "C" / "derivatives"
EPOCHS_COMBINED_C = DERIV_C / "epochs_combined_C-epo.fif"
EPO_RUN_C = BASE / "C" / "epochs_with_run_C-epo.fif"
EPO_MASTER_C = DERIV_C / "epochs_with_master_C-epo.fif"
RUNS_C_DIR = DERIV_C  # folder with run1, run2, ...

# S
DERIV_S = BASE / "S" / "derivatives"
EPOCHS_COMBINED_S = DERIV_S / "epochs_combined_S-epo.fif"
EPO_RUN_S = BASE / "S" / "epochs_with_run_S-epo.fif"
EPO_MASTER_S = DERIV_S / "epochs_with_master_S-epo.fif"
RUNS_S_DIR = DERIV_S  # folder with run1, run2, ...

MASTER_CSV = BEHAV_DIR / "master_behavior.csv"
MASTER_FB_CSV = BEHAV_DIR / "master_behavior_fb_unique.csv"
MASTER_PROBE_CSV = BEHAV_DIR / "master_behavior_with_probe.csv"

EPOCHS_PATTERN = "*epochs-epo.fif"

print("=" * 70)
print(f"CONFIGURATION — Subject: {SUBJECT}")
print("=" * 70)
print(f"\nBASE:      {BASE}")
print(f"BEHAV_DIR: {BEHAV_DIR}")
print(f"DERIV_C:   {DERIV_C}")
print(f"DERIV_S:   {DERIV_S}")

print(f"\nINPUT FILES:")
print(f"   {'✓' if EPOCHS_COMBINED_C.exists() else '✗'} epochs_combined_C-epo.fif")
print(f"   {'✓' if EPOCHS_COMBINED_S.exists() else '✗'} epochs_combined_S-epo.fif")
print(f"   {'✓' if list(BEHAV_DIR.glob('Mnemosyne_*_run*.mat')) else '✗'} Mnemosyne_*_run*.mat")

print(f"\nOUTPUT FILES:")
print(f"master_behavior.csv")
print(f"master_behavior_fb_unique.csv")
print(f"master_behavior_with_probe.csv")
print(f"epochs_with_run_C/S-epo.fif")
print(f"epochs_with_master_C/S-epo.fif")
print("=" * 70)


In [ ]:

# STEP 1: behaviour out of Mnemosyne_*.mat → makes master_behavior.csv

def load_mnemo_matrix(path: Path) -> pd.DataFrame:
    mat = sio.loadmat(path, squeeze_me=True, struct_as_record=False)
    data = mat.get("data", None)
    if data is None or not hasattr(data, "matrix"):
        raise RuntimeError(f"{path.name}: data.matrix not found")
    M = np.asarray(data.matrix)
    assert M.ndim == 2 and M.shape[1] == 17, f"{path.name}: expected shape (N, 17), got {M.shape}"

    cols = ["episode", "taskset", "stimulus", "expected_resp", "trap_flag",
            "jitter1", "jitter2", "run", "post_pause",
            "code", "keycode", "cor", "rew", "rt",
            "tempo_stim", "tempo_reward", "tempo_trial"]
    df = pd.DataFrame(M, columns=cols)

    block = "C" if "_C" in path.name else "S"
    df.insert(0, "trial_in_file", np.arange(1, len(df) + 1))
    df.insert(0, "run", df.pop("run").astype(int))
    df.insert(0, "block", block)
    df.insert(0, "file", path.name)

    for c in ["episode", "taskset", "stimulus", "expected_resp", "trap_flag",
              "post_pause", "code", "keycode", "cor", "rew"]:
        if c in df.columns:
            df[c] = df[c].astype(int)

    return df


# Load all mat files
mnemo_files = sorted(BEHAV_DIR.glob("Mnemosyne_*_run*.mat"))
assert mnemo_files, f"No Mnemosyne_*_run*.mat in {BEHAV_DIR}"

tables = [load_mnemo_matrix(p) for p in mnemo_files]
master = pd.concat(tables, ignore_index=True)

# Sort and assign sequential trial numbers within each (block, run)
master.sort_values(["block", "run", "file", "trial_in_file"], inplace=True)
master["trial"] = master.groupby(["block", "run"], sort=False).cumcount() + 1

# Sanity check: (block, run, trial) must be unique
assert master.duplicated(["block", "run", "trial"]).sum() == 0, "Duplicates found in (block,run,trial)"

keep = ["file", "block", "run", "trial",
        "episode", "taskset", "stimulus", "expected_resp", "trap_flag",
        "jitter1", "jitter2", "post_pause",
        "code", "keycode", "cor", "rew", "rt",
        "tempo_stim", "tempo_reward", "tempo_trial"]
master = master[keep].copy()

master.to_csv(MASTER_CSV, index=False, encoding="utf-8")
print(f"✓ Saved: {MASTER_CSV}")
print(f"  Rows: {len(master)}, Blocks: {master['block'].unique().tolist()}")


In [ ]:

# STEP 2: Add run/trial to epochs_combined → epochs_with_run

import gc

def _sort_runs_numerically(paths):
    def key(p):
        m = re.search(r"run*([0-9]+)", p.name)
        return int(m.group(1)) if m else p.name
    return sorted(paths, key=key)


def load_run_epochs(run_dir: Path, pattern=EPOCHS_PATTERN):
    files = sorted(run_dir.glob(pattern))
    assert files, f"No epoch files in {run_dir} matching {pattern}"
    return mne.concatenate_epochs(
        [mne.read_epochs(str(p), preload=False, verbose="ERROR") for p in files]
    )


def find_run_order(eps_combined, per_run_dirs, pattern=EPOCHS_PATTERN):
    per_run = [(Path(d), load_run_epochs(Path(d), pattern=pattern)) for d in per_run_dirs]

    ordered_dirs = []
    ordered_counts = []
    unmatched_idx = list(range(len(per_run)))
    combined_pos = 0

    for slot in range(len(per_run)):
        matched = False
        for j in unmatched_idx:
            d, er = per_run[j]
            n = len(er)
            if combined_pos + n > len(eps_combined):
                continue
            a = eps_combined.events[combined_pos:combined_pos + n, 2]
            b = er.events[:, 2]
            if np.array_equal(a, b):
                ordered_dirs.append(d)
                ordered_counts.append(n)
                unmatched_idx.remove(j)
                combined_pos += n
                matched = True
                print(f"  slot {slot+1} → {d.name} ({n} epochs)")
                break
        if not matched:
            print(f"\n[DIAGNOSTICS] slot {slot+1}: {combined_pos}")
            print(f"  Combined events[{combined_pos}:{combined_pos+5}]: "
                  f"{eps_combined.events[combined_pos:combined_pos+5, 2]}")
            for j in unmatched_idx:
                d2, er2 = per_run[j]
                print(f"  {d2.name} events[:5]: {er2.events[:5, 2]}")
            raise RuntimeError(
                f"Failed to match run at position {combined_pos} in combined file.\n"
                f"Check: the combined file may contain filtered trials "
                f"that are not present in the per-run files."
            )

    if combined_pos != len(eps_combined):
        raise RuntimeError(f"Only matched {combined_pos} out of {len(eps_combined)} epochs.")

    return ordered_dirs, ordered_counts


def inject_run_into_combined(eps_combined, ordered_dirs, counts, start_index=1):
    """Injects run/trial into metadata based on known counts."""
    run_vec = np.concatenate([
        np.full(n, i + start_index, dtype=int) for i, n in enumerate(counts)
    ])
    assert len(run_vec) == len(eps_combined)

    md = (eps_combined.metadata.copy() if eps_combined.metadata is not None
          else pd.DataFrame(index=np.arange(len(eps_combined))))
    md["run"] = run_vec
    md["trial"] = md.groupby("run", sort=False).cumcount() + 1
    eps_combined.metadata = md
    return eps_combined


# Find run* folders
RUNS_C = _sort_runs_numerically(list(RUNS_C_DIR.glob("run*")))
RUNS_S = _sort_runs_numerically(list(RUNS_S_DIR.glob("run*")))

print(f"[INFO] runs C: {[r.name for r in RUNS_C]}")
print(f"[INFO] runs S: {[r.name for r in RUNS_S]}\n")

# Load combined epochs
epochs_C = mne.read_epochs(str(EPOCHS_COMBINED_C), preload=False, verbose="ERROR")
epochs_S = mne.read_epochs(str(EPOCHS_COMBINED_S), preload=False, verbose="ERROR")

# Match each run by event sequence to recover the actual order in the combined file
print("[C] Matching runs with combined_C...")
ordered_C, counts_C = find_run_order(epochs_C, RUNS_C, pattern=EPOCHS_PATTERN)

print("\n[S] Matching runs with combined_S...")
ordered_S, counts_S = find_run_order(epochs_S, RUNS_S, pattern=EPOCHS_PATTERN)

print(f"\n[INFO] Actual order C: {[d.name for d in ordered_C]}, counts={counts_C}")
print(f"[INFO] Actual order S: {[d.name for d in ordered_S]}, counts={counts_S}")

# Warn if the order differs from numerical
if [d.name for d in ordered_C] != [d.name for d in RUNS_C]:
    print(f"\n⚠  C: order of runs in combined differs from numerical!")
    print(f"   numerical:   {[d.name for d in RUNS_C]}")
    print(f"   actual: {[d.name for d in ordered_C]}")
if [d.name for d in ordered_S] != [d.name for d in RUNS_S]:
    print(f"\n⚠  S: order of runs in combined differs from numerical!")
    print(f"   numerical:   {[d.name for d in RUNS_S]}")
    print(f"   actual: {[d.name for d in ordered_S]}")

# Inject run/trial based on actual order
epochs_C = inject_run_into_combined(epochs_C, ordered_C, counts_C)
epochs_S = inject_run_into_combined(epochs_S, ordered_S, counts_S)


def _cleanup_epoch_splits(base_path: Path):
    base_path = Path(base_path)
    files = [base_path, *base_path.parent.glob(f"{base_path.stem}-*{base_path.suffix}")]
    for fp in sorted(set(files)):
        try:
            fp.unlink()
        except FileNotFoundError:
            pass
        except PermissionError as e:
            print(f"[WARN] cannot remove locked file: {fp.name} ({e})")


def save_epochs_robust(epochs_obj, out_path: Path, split_sizes=("100MB", "50MB", "20MB")):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    epochs_obj.load_data()

    last_err = None
    for split_size in split_sizes:
        try:
            _cleanup_epoch_splits(out_path)
            epochs_obj.save(str(out_path), overwrite=True, split_size=split_size, fmt="single")
            mne.read_epochs(str(out_path), preload=False, verbose="ERROR")
            print(f"[OK] saved: {out_path} (split_size={split_size})")
            return out_path
        except Exception as e:
            last_err = e
            print(f"[WARN] save failed for {out_path.name} (split_size={split_size}): {type(e).__name__}: {e}")

    raise RuntimeError(f"Failed to save epochs after retries: {out_path}") from last_err


EPO_RUN_C = save_epochs_robust(epochs_C, EPO_RUN_C)
del epochs_C
gc.collect()

EPO_RUN_S = save_epochs_robust(epochs_S, EPO_RUN_S)
print(f"✓ Saved: {EPO_RUN_C}")
print(f"✓ Saved: {EPO_RUN_S}")


In [ ]:
# STEP 3: Build master_behavior_fb_unique.csv (feedback rows aligned to MEG)


import re

master = pd.read_csv(MASTER_CSV)
assert "tempo_reward" in master.columns, "Missing tempo_reward in master_behavior.csv"


def _sort_runs_numerically(paths):
    """Sort run1, run2, ... folders numerically."""
    def key(p):
        m = re.search(r"run0*([0-9]+)", p.name, flags=re.IGNORECASE)
        return int(m.group(1)) if m else p.name
    return sorted(paths, key=key)


def _discover_run_dirs(primary: Path, deriv: Path, block_root: Path):
    """Find run* folders with fallback across likely roots."""
    for root in (primary, deriv, block_root):
        if root is None:
            continue
        root = Path(root)
        if not root.exists():
            continue
        runs = [d for d in root.glob("run*") if d.is_dir()]
        runs = _sort_runs_numerically(runs)
        if runs:
            return runs, root
    return [], None


def load_run_meg_feedback(run_dir: Path, block_label: str, pattern=EPOCHS_PATTERN) -> pd.DataFrame:
    """Load feedback metadata rows from run-level epochs files."""
    m = re.search(r"run0*([0-9]+)", run_dir.name, flags=re.IGNORECASE)
    if not m:
        raise RuntimeError(f"{block_label}: cannot parse run number from {run_dir.name}")
    run_num = int(m.group(1))

    epo_files = sorted(run_dir.glob(pattern))
    if not epo_files:
        raise RuntimeError(f"{block_label}: no files matching {pattern} in {run_dir}")

    parts = []
    for epo_path in epo_files:
        ep = mne.read_epochs(str(epo_path), preload=False, verbose="ERROR")

        if ep.metadata is None:
            # Fallback: epochs saved without metadata — reconstruct onset_s from events
            print(f"[WARN] {block_label}/{run_dir.name}/{epo_path.name}: no metadata, "
                  f"building onset_s from events (n={len(ep)})")
            md = pd.DataFrame({"onset_s": ep.events[:, 0] / ep.info["sfreq"]})
        else:
            md = ep.metadata.copy().reset_index(drop=True)
            if "onset_s" not in md.columns:
                md["onset_s"] = ep.events[:, 0] / ep.info["sfreq"]

        md["onset_s"] = pd.to_numeric(md["onset_s"], errors="coerce")
        if "stim_onset_s" in md.columns:
            md["stim_onset_s"] = pd.to_numeric(md["stim_onset_s"], errors="coerce")
        elif "stim_to_fb_s" in md.columns:
            md["stim_to_fb_s"] = pd.to_numeric(md["stim_to_fb_s"], errors="coerce")
            md["stim_onset_s"] = md["onset_s"] - md["stim_to_fb_s"]

        parts.append(md)

    md = pd.concat(parts, ignore_index=True)
    md["block"] = block_label
    md["run"] = run_num
    md["meta_row"] = np.arange(1, len(md) + 1)
    if "trial" not in md.columns:
        md["trial"] = md["meta_row"]
    return md


# Prepare behavior table (deduplicate repeated matrix rows by reward onset)
beh = master.copy()
beh["block"] = beh["block"].astype(str)
beh["run"] = pd.to_numeric(beh["run"], errors="coerce")
beh["tempo_reward"] = pd.to_numeric(beh["tempo_reward"], errors="coerce")
if "tempo_stim" in beh.columns:
    beh["tempo_stim"] = pd.to_numeric(beh["tempo_stim"], errors="coerce")

beh = beh[
    beh["run"].notna()
    & beh["tempo_reward"].notna()
    & (beh["tempo_reward"] > 0)
].copy()
beh["run"] = beh["run"].astype(int)
beh = beh.sort_values(["block", "run", "tempo_reward"]).drop_duplicates(
    ["block", "run", "tempo_reward"], keep="first"
).reset_index(drop=True)


# Discover run folders
RUNS_C_DIR = Path(globals().get("RUNS_C_DIR", DERIV_C))
RUNS_S_DIR = Path(globals().get("RUNS_S_DIR", DERIV_S))
RUNS_C, RUNS_C_SRC = _discover_run_dirs(RUNS_C_DIR, DERIV_C, BASE / "C")
RUNS_S, RUNS_S_SRC = _discover_run_dirs(RUNS_S_DIR, DERIV_S, BASE / "S")

print(f"[INFO] C runs source: {RUNS_C_SRC}")
print(f"[INFO] S runs source: {RUNS_S_SRC}")
print(f"[INFO] Found runs C: {[d.name for d in RUNS_C]}")
print(f"[INFO] Found runs S: {[d.name for d in RUNS_S]}")

if not RUNS_C and not RUNS_S:
    raise RuntimeError("No run* folders found for C/S in configured locations")


# Build MEG-anchored table: one row per MEG feedback epoch
meg_parts = []
for d in RUNS_C:
    meg_parts.append(load_run_meg_feedback(d, "C"))
for d in RUNS_S:
    meg_parts.append(load_run_meg_feedback(d, "S"))

meg_tbl = pd.concat(meg_parts, ignore_index=True)
meg_tbl = meg_tbl[meg_tbl["onset_s"].notna()].copy()


# Match behavior to MEG rows run-by-run using robust offset correction
matched_parts = []
for (block, run), md_r in meg_tbl.groupby(["block", "run"], sort=False):
    md_r = md_r.sort_values("onset_s").reset_index(drop=True).copy()
    beh_r = beh[(beh["block"] == block) & (beh["run"] == int(run))].copy()

    if beh_r.empty:
        md_r["offset_s"] = np.nan
        md_r["dt_reward"] = np.nan
        md_r["match_ok"] = False
        matched_parts.append(md_r)
        continue

    beh_r = beh_r.sort_values("tempo_reward").reset_index(drop=True)

    # Align clocks by per-run median offset before nearest matching.
    offset_s = float(md_r["onset_s"].median() - beh_r["tempo_reward"].median())
    md_r["onset_adj"] = md_r["onset_s"] - offset_s

    beh_keep = [c for c in beh_r.columns if c not in ("block", "run")]
    merged = pd.merge_asof(
        md_r.sort_values("onset_adj"),
        beh_r[beh_keep].sort_values("tempo_reward"),
        left_on="onset_adj",
        right_on="tempo_reward",
        direction="nearest",
    )

    merged["offset_s"] = offset_s
    merged["dt_reward"] = (merged["onset_adj"] - merged["tempo_reward"]).abs()

    if "stim_onset_s" in merged.columns and "tempo_stim" in merged.columns:
        merged["dt_stim"] = ((merged["stim_onset_s"] - offset_s) - merged["tempo_stim"]).abs()

    matched_parts.append(merged)

beh_fb_unique = pd.concat(matched_parts, ignore_index=True)

# Diagnostic quality flag (not used as hard filter by default)
MAX_DT_S = 2.0
if MAX_DT_S is None:
    beh_fb_unique["match_ok"] = True
else:
    beh_fb_unique["match_ok"] = beh_fb_unique["dt_reward"] <= MAX_DT_S
    if "dt_stim" in beh_fb_unique.columns:
        beh_fb_unique["match_ok"] &= beh_fb_unique["dt_stim"] <= MAX_DT_S

KEEP_ONLY_MATCH_OK = False
if KEEP_ONLY_MATCH_OK:
    beh_fb_unique = beh_fb_unique[beh_fb_unique["match_ok"]].copy()

beh_fb_unique = beh_fb_unique.sort_values(["block", "run", "onset_s"]).reset_index(drop=True)
beh_fb_unique.to_csv(MASTER_FB_CSV, index=False, encoding="utf-8")
print(f"Saved: {MASTER_FB_CSV}")

# QC report
print("[REPORT]")
print(f"  master rows:              {len(master)}")
print(f"  behavior dedup rows:      {len(beh)}")
print(f"  MEG feedback rows:        {len(meg_tbl)}")
print(f"  saved rows:               {len(beh_fb_unique)}")
print(f"  match_ok (<= {MAX_DT_S}s): {int(beh_fb_unique['match_ok'].sum())} / {len(beh_fb_unique)}")

if len(beh_fb_unique) > 0 and "dt_reward" in beh_fb_unique.columns:
    print(
        f"  dt_reward (s): mean={beh_fb_unique['dt_reward'].mean():.4f} | "
        f"median={beh_fb_unique['dt_reward'].median():.4f} | "
        f"p95={beh_fb_unique['dt_reward'].quantile(0.95):.4f}"
    )
    if "dt_stim" in beh_fb_unique.columns:
        print(
            f"  dt_stim   (s): mean={beh_fb_unique['dt_stim'].mean():.4f} | "
            f"median={beh_fb_unique['dt_stim'].median():.4f} | "
            f"p95={beh_fb_unique['dt_stim'].quantile(0.95):.4f}"
        )

print(f"  MEG rows per block:       {meg_tbl.groupby('block').size().to_dict()}")
print(f"  Saved rows per block:     {beh_fb_unique.groupby('block').size().to_dict()}")
print(f"  match_ok per block:       {beh_fb_unique.groupby('block')['match_ok'].sum().to_dict()}")


In [ ]:
# PROBE (fit) + RL comparison


import importlib
import back_model as bm

bm = importlib.reload(bm)
prepare_behavior_arrays = bm.prepare_behavior_arrays
fit_probe_full = bm.fit_probe_full
fit_probe_mcmc = getattr(bm, "fit_probe_mcmc", None)
fit_probe_shared_single_tau = getattr(bm, "fit_probe_shared_single_tau", None)
fit_probe_confirmatory = getattr(bm, "fit_probe_confirmatory", None)
fit_probe_shared_core_separate_beta_eps = getattr(bm, "fit_probe_shared_core_separate_beta_eps", None)
fit_rl = bm.fit_rl
attach_probe_regressors = bm.attach_probe_regressors
probe_qc_metrics = bm.probe_qc_metrics

import mne
import numpy as np
import pandas as pd
import time as _time

def _ts():
    return _time.strftime("%H:%M:%S")

def _elapsed(t0):
    dt = _time.time() - t0
    return f"{dt:.1f}s" if dt < 60 else f"{dt/60:.1f}min"

MASTER_FB = MASTER_FB_CSV
MODEL_OUT = BEHAV_DIR
MODEL_OUT.mkdir(exist_ok=True)

# Column mapping (edit if your data use different names)
STIM_COL = "stimulus"
ACTION_COL = "code"
OUTCOME_COL = "rew"
POSITIVE_OUTCOME_VALUE = 1
HAZARD_COL = "taskset"     # column encoding the active task-set (used to estimate switch rate)

if ACTION_COL not in pd.read_csv(MASTER_FB, nrows=1).columns:
    ACTION_COL = "keycode"
if OUTCOME_COL not in pd.read_csv(MASTER_FB, nrows=1).columns:
    OUTCOME_COL = "cor"

N_RESTARTS = 30
FIT_N_JOBS = globals().get("FIT_N_JOBS", None)
FIT_METHOD = str(globals().get("FIT_METHOD", "mle")).strip().lower()
FIT_TAU_MODE = "confirmatory"  # separate | confirmatory | beta_eps_separate
TAU_SHARED_BOUNDS = tuple(globals().get("TAU_SHARED_BOUNDS", (0.0, 1.0)))

if FIT_METHOD == "mcmc" and fit_probe_mcmc is None:
    print("[WARN] fit_probe_mcmc is unavailable in current back_model; fallback to FIT_METHOD='mle'.")
    FIT_METHOD = "mle"
valid_tau_modes = {"separate", "confirmatory", "beta_eps_separate"}
if FIT_TAU_MODE not in valid_tau_modes:
    print(f"[WARN] unknown FIT_TAU_MODE='{FIT_TAU_MODE}', fallback to 'confirmatory'.")
    FIT_TAU_MODE = "confirmatory"
if FIT_TAU_MODE == "confirmatory" and (fit_probe_confirmatory is None and fit_probe_shared_single_tau is None):
    print("[WARN] confirmatory fit is unavailable in current back_model; fallback to FIT_TAU_MODE='separate'.")
    FIT_TAU_MODE = "separate"
if FIT_TAU_MODE == "beta_eps_separate" and fit_probe_shared_core_separate_beta_eps is None:
    print("[WARN] beta_eps_separate fit is unavailable in current back_model; fallback to FIT_TAU_MODE='confirmatory'.")
    FIT_TAU_MODE = "confirmatory"
if FIT_TAU_MODE in {"confirmatory", "beta_eps_separate"} and FIT_METHOD == "mcmc":
    print(f"[WARN] FIT_TAU_MODE='{FIT_TAU_MODE}' currently supports optimizer fit only; fallback to FIT_METHOD='mle'.")
    FIT_METHOD = "mle"

MCMC_N_SAMPLES = int(globals().get("MCMC_N_SAMPLES", 20000))
MCMC_BURN_IN = int(globals().get("MCMC_BURN_IN", 5000))
MCMC_THIN = int(globals().get("MCMC_THIN", 5))
MCMC_N_RESAMPLES = int(globals().get("MCMC_N_RESAMPLES", 300))


def model_stats(ll: float, k: int, n: int) -> dict:
    aic = 2 * k - 2 * ll
    bic = k * np.log(max(n, 1)) - 2 * ll
    return {"ll": ll, "aic": aic, "bic": bic}


def _estimate_hazard_from_protocol(df_block: pd.DataFrame, hazard_col: str = HAZARD_COL) -> float:
    if hazard_col not in df_block.columns:
        raise RuntimeError(f"Cannot estimate hazard: missing column '{hazard_col}'.")

    if "run" in df_block.columns:
        d = df_block.sort_values(["run", "trial"] if "trial" in df_block.columns else ["run"]).copy()
    else:
        d = df_block.copy().reset_index(drop=True)
        d["run"] = 1

    d[hazard_col] = pd.to_numeric(d[hazard_col], errors="coerce")
    d = d.dropna(subset=[hazard_col]).copy()
    if d.empty:
        raise RuntimeError(f"Cannot estimate hazard: no finite values in '{hazard_col}'.")

    total_switches = 0
    total_pairs = 0
    for _, g in d.groupby("run", sort=False):
        vals = g[hazard_col].to_numpy()
        if vals.size <= 1:
            continue
        total_switches += int(np.sum(vals[1:] != vals[:-1]))
        total_pairs += int(vals.size - 1)

    if total_pairs <= 0:
        raise RuntimeError("Cannot estimate hazard: not enough trials per run.")
    return float(total_switches / float(total_pairs))


def fit_block(df_block: pd.DataFrame, block_label: str, n_restarts: int = N_RESTARTS, n_jobs=FIT_N_JOBS) -> tuple:
    df_block = df_block.reset_index(drop=True)
    s, a, o, S, A, O, maps = prepare_behavior_arrays(
        df_block, STIM_COL, ACTION_COL, OUTCOME_COL,
        positive_outcome_value=POSITIVE_OUTCOME_VALUE,
        require_positive_outcome_value=True,
    )
    positive_outcome_idx = int(maps.get("positive_outcome_idx", O - 1))

    if FIT_METHOD == "mcmc":
        print(f"[{_ts()}] [{block_label}] Fitting PROBE (MCMC, {MCMC_N_SAMPLES} samples, burn={MCMC_BURN_IN})...", flush=True)
        _t0_probe = _time.time()
        p_hat, out_probe, probe_meta = fit_probe_mcmc(
            s, a, o, S, A, O,
            N=3,
            n_samples=MCMC_N_SAMPLES,
            burn_in=MCMC_BURN_IN,
            thin=MCMC_THIN,
            n_resamples=MCMC_N_RESAMPLES,
            positive_outcome_idx=positive_outcome_idx,
            seed=7,
        )
    else:
        print(f"[{_ts()}] [{block_label}] Fitting PROBE (MLE, {n_restarts} restarts)...", flush=True)
        _t0_probe = _time.time()
        p_hat, out_probe, probe_meta = fit_probe_full(
            s, a, o, S, A, O, N=3, n_restarts=n_restarts, seed=7,
            positive_outcome_idx=positive_outcome_idx, n_jobs=n_jobs
        )
    print(f"[{_ts()}] [{block_label}] PROBE done ({_elapsed(_t0_probe)})  LL={float(out_probe['ll']):.2f}  tau={getattr(p_hat,'tau',float('nan')):.4f}  eta={getattr(p_hat,'eta',float('nan')):.4f}", flush=True)

    print(f"[{_ts()}] [{block_label}] Fitting RL ({n_restarts} restarts)...", flush=True)
    _t0_rl = _time.time()
    rl_hat, rl_ll, _ = fit_rl(
        s, a, o, S, A, n_restarts=n_restarts, seed=7,
        positive_outcome_idx=positive_outcome_idx, n_jobs=n_jobs
    )
    print(f"[{_ts()}] [{block_label}] RL done ({_elapsed(_t0_rl)})  LL={float(rl_ll):.2f}  alpha={getattr(rl_hat,'alpha',float('nan')):.4f}  beta={getattr(rl_hat,'beta',float('nan')):.4f}", flush=True)

    df_out = attach_probe_regressors(df_block, out_probe)
    return df_out, p_hat, out_probe, probe_meta, rl_hat, rl_ll, positive_outcome_idx


# Load behavior (feedback-aligned)
beh_fb_raw = pd.read_csv(MASTER_FB)
beh_fb_raw["block"] = beh_fb_raw["block"].astype(str)
beh_fb = beh_fb_raw.copy()

# Keep only valid trials for model fitting (exclude omissions/invalid feedback).
beh_fb[ACTION_COL] = pd.to_numeric(beh_fb[ACTION_COL], errors="coerce")
beh_fb[OUTCOME_COL] = pd.to_numeric(beh_fb[OUTCOME_COL], errors="coerce")
beh_fb = beh_fb[(beh_fb[ACTION_COL] > 0) & (beh_fb[OUTCOME_COL] >= 0)].copy()

# QC: check whether action distribution is near-random (e.g., ~50/50 for binary choices)
for block_qc in ["C", "S"]:
    df_qc = beh_fb[beh_fb["block"] == block_qc].copy()
    if df_qc.empty:
        continue
    if STIM_COL in df_qc.columns and ACTION_COL in df_qc.columns:
        qc = (
            df_qc.groupby([STIM_COL, ACTION_COL], dropna=False)
            .size()
            .rename("n")
            .reset_index()
        )
        qc["p"] = qc.groupby(STIM_COL)["n"].transform(lambda x: x / x.sum())
        print(f"[QC] block={block_qc} choice proportions by stimulus")
        print(qc.sort_values([STIM_COL, ACTION_COL]).to_string(index=False))

blocks = ["C", "S"]
all_rows = []
fit_rows = []

_cell_t0 = _time.time()
_n_c = len(beh_fb[beh_fb["block"] == "C"])
_n_s = len(beh_fb[beh_fb["block"] == "S"])
print(f"\n[{_ts()}] Starting model fitting")
print(f"         method={FIT_METHOD!r}  tau_mode={FIT_TAU_MODE!r}  restarts={N_RESTARTS}  jobs={FIT_N_JOBS}")
print(f"         trials: C={_n_c}  S={_n_s}  total={_n_c + _n_s}")

joint_modes = {"confirmatory", "beta_eps_separate"}
if FIT_TAU_MODE in joint_modes:
    df_s = beh_fb[beh_fb["block"] == "S"].copy().reset_index(drop=True)
    df_c = beh_fb[beh_fb["block"] == "C"].copy().reset_index(drop=True)
    if df_s.empty or df_c.empty:
        raise RuntimeError(f"FIT_TAU_MODE='{FIT_TAU_MODE}' requires both S and C blocks with non-empty data.")

    s_s, a_s, o_s, S_s, A_s, O_s, maps_s = prepare_behavior_arrays(
        df_s, STIM_COL, ACTION_COL, OUTCOME_COL,
        positive_outcome_value=POSITIVE_OUTCOME_VALUE,
        require_positive_outcome_value=True,
    )
    s_c, a_c, o_c, S_c, A_c, O_c, maps_c = prepare_behavior_arrays(
        df_c, STIM_COL, ACTION_COL, OUTCOME_COL,
        positive_outcome_value=POSITIVE_OUTCOME_VALUE,
        require_positive_outcome_value=True,
    )

    pos_idx_s = int(maps_s.get("positive_outcome_idx", O_s - 1))
    pos_idx_c = int(maps_c.get("positive_outcome_idx", O_c - 1))

    tau_env_s = np.nan
    tau_env_c = np.nan
    k_bias = np.nan
    probe_k = 7

    if FIT_TAU_MODE == "confirmatory":
        fit_fn = fit_probe_confirmatory if fit_probe_confirmatory is not None else fit_probe_shared_single_tau
        print(f"[{_ts()}] Joint PROBE fit ({FIT_TAU_MODE}, {N_RESTARTS} restarts, C+S)...", flush=True)
        _t0_probe_joint = _time.time()
        p_hat_s, out_s, p_hat_c, out_c, probe_meta_shared = fit_fn(
            s_s, a_s, o_s, S_s, A_s, O_s,
            s_c, a_c, o_c, S_c, A_c, O_c,
            N=3,
            n_restarts=N_RESTARTS,
            seed=7,
            positive_outcome_idx_s=pos_idx_s,
            positive_outcome_idx_c=pos_idx_c,
            tau_bounds=TAU_SHARED_BOUNDS,
            n_jobs=FIT_N_JOBS,
        )
        probe_k = 6

    else:  # beta_eps_separate
        print(f"[{_ts()}] Joint PROBE fit ({FIT_TAU_MODE}, {N_RESTARTS} restarts, C+S)...", flush=True)
        _t0_probe_joint = _time.time()
        p_hat_s, out_s, p_hat_c, out_c, probe_meta_shared = fit_probe_shared_core_separate_beta_eps(
            s_s, a_s, o_s, S_s, A_s, O_s,
            s_c, a_c, o_c, S_c, A_c, O_c,
            N=3,
            n_restarts=N_RESTARTS,
            seed=7,
            positive_outcome_idx_s=pos_idx_s,
            positive_outcome_idx_c=pos_idx_c,
            tau_bounds=TAU_SHARED_BOUNDS,
            n_jobs=FIT_N_JOBS,
        )
        probe_k = 8

    print(f"[{_ts()}] Joint PROBE done ({_elapsed(_t0_probe_joint)})  LL_C={float(out_c['ll']):.2f}  LL_S={float(out_s['ll']):.2f}", flush=True)
    print(f"         tau={getattr(p_hat_c,'tau',float('nan')):.4f}  eta_C={getattr(p_hat_c,'eta',float('nan')):.4f}  eta_S={getattr(p_hat_s,'eta',float('nan')):.4f}", flush=True)

    print(f"[{_ts()}] Fitting RL block=C ({N_RESTARTS} restarts)...", flush=True)
    _t0_rl_c = _time.time()
    rl_hat_c, rl_ll_c, _ = fit_rl(
        s_c, a_c, o_c, S_c, A_c,
        n_restarts=N_RESTARTS,
        seed=7,
        positive_outcome_idx=pos_idx_c,
        n_jobs=FIT_N_JOBS,
    )
    print(f"[{_ts()}] RL[C] done ({_elapsed(_t0_rl_c)})  LL={float(rl_ll_c):.2f}  alpha={getattr(rl_hat_c,'alpha',float('nan')):.4f}", flush=True)

    print(f"[{_ts()}] Fitting RL block=S ({N_RESTARTS} restarts)...", flush=True)
    _t0_rl_s = _time.time()
    rl_hat_s, rl_ll_s, _ = fit_rl(
        s_s, a_s, o_s, S_s, A_s,
        n_restarts=N_RESTARTS,
        seed=7,
        positive_outcome_idx=pos_idx_s,
        n_jobs=FIT_N_JOBS,
    )
    print(f"[{_ts()}] RL[S] done ({_elapsed(_t0_rl_s)})  LL={float(rl_ll_s):.2f}  alpha={getattr(rl_hat_s,'alpha',float('nan')):.4f}", flush=True)

    df_out_c = attach_probe_regressors(df_c, out_c)
    df_out_s = attach_probe_regressors(df_s, out_s)
    all_rows.extend([df_out_c, df_out_s])

    n_c = len(df_out_c)
    n_s = len(df_out_s)
    ll_c = float(out_c["ll"])
    ll_s = float(out_s["ll"])
    ll_combined = ll_c + ll_s
    n_combined = n_c + n_s

    probe_stats = model_stats(ll_combined, k=probe_k, n=n_combined)

    joint_results = [
        ("C", df_out_c, p_hat_c, out_c, probe_meta_shared, rl_hat_c, rl_ll_c, tau_env_c, n_c),
        ("S", df_out_s, p_hat_s, out_s, probe_meta_shared, rl_hat_s, rl_ll_s, tau_env_s, n_s),
    ]
    for block, df_out, p_hat, out_probe, probe_meta, rl_hat, rl_ll, tau_env_block, n_block in joint_results:
        rl_stats = model_stats(float(rl_ll), k=3, n=n_block)
        probe_qc = probe_qc_metrics(out_probe)

        fit_rows.append({
            "block": block,
            "model": "PROBE",
            "fit_method": FIT_METHOD,
            "fit_tau_mode": FIT_TAU_MODE,
            "n": n_block,
            "tau": p_hat.tau,
            "tau_env": tau_env_block,
            "k_bias": k_bias,
            "eta": p_hat.eta,
            "theta": p_hat.theta,
            "alpha_q": p_hat.alpha_q,
            "beta": p_hat.beta,
            "eps": p_hat.eps,
            "ll_block": float(out_probe["ll"]),    # per-block LL for diagnostics; ll/aic/bic below are combined
            "rho_counts": getattr(p_hat, "rho_counts", np.nan),
            "switch_entropy_threshold": getattr(p_hat, "switch_entropy_threshold", np.nan),
            "switch_min_streak": getattr(p_hat, "switch_min_streak", np.nan),
            "mcmc_accept_rate": (probe_meta.get("accept_rate", np.nan) if isinstance(probe_meta, dict) else np.nan),
            **probe_stats,
            **probe_qc,
        })
        fit_rows.append({
            "block": block,
            "model": "RL",
            "fit_method": FIT_METHOD,
            "fit_tau_mode": FIT_TAU_MODE,
            "n": n_block,
            "alpha": rl_hat.alpha,
            "beta": rl_hat.beta,
            "eps": rl_hat.eps,
            **rl_stats,
            "probe_active_mean": np.nan,
            "p_lam_actor_gt_0.5": np.nan,
            "n_confirm": np.nan,
            "n_reject": np.nan,
            "n_switch_in": np.nan,
            "degenerate_probe": np.nan,
            "mcmc_accept_rate": np.nan,
        })

else:
    for block in blocks:
        dfb = beh_fb[beh_fb["block"] == block].copy()
        if dfb.empty:
            print(f"[WARN] No rows for block {block}")
            continue

        print(f"\n[{_ts()}] Block={block!r} (separate mode, {N_RESTARTS} restarts)", flush=True)
        _t0_block = _time.time()

        df_out, p_hat, out_probe, probe_meta, rl_hat, rl_ll, _ = fit_block(
            dfb, block, n_restarts=N_RESTARTS, n_jobs=FIT_N_JOBS
        )
        all_rows.append(df_out)

        n = len(df_out)
        probe_stats = model_stats(float(out_probe["ll"]), k=6, n=n)
        rl_stats = model_stats(float(rl_ll), k=3, n=n)
        probe_qc = probe_qc_metrics(out_probe)

        print(f"[{_ts()}] Block={block!r} complete ({_elapsed(_t0_block)})  ΔAIC(PROBE-RL)={probe_stats['aic'] - rl_stats['aic']:.1f}", flush=True)

        fit_rows.append({
            "block": block,
            "model": "PROBE",
            "fit_method": FIT_METHOD,
            "fit_tau_mode": FIT_TAU_MODE,
            "n": n,
            "tau": p_hat.tau,
            "tau_env": np.nan,
            "k_bias": np.nan,
            "eta": p_hat.eta,
            "theta": p_hat.theta,
            "alpha_q": p_hat.alpha_q,
            "beta": p_hat.beta,
            "eps": p_hat.eps,
            "ll_block": float(out_probe["ll"]),
            "rho_counts": getattr(p_hat, "rho_counts", np.nan),
            "switch_entropy_threshold": getattr(p_hat, "switch_entropy_threshold", np.nan),
            "switch_min_streak": getattr(p_hat, "switch_min_streak", np.nan),
            "mcmc_accept_rate": (probe_meta.get("accept_rate", np.nan) if isinstance(probe_meta, dict) else np.nan),
            **probe_stats,
            **probe_qc,
        })
        fit_rows.append({
            "block": block,
            "model": "RL",
            "fit_method": FIT_METHOD,
            "fit_tau_mode": FIT_TAU_MODE,
            "n": n,
            "alpha": rl_hat.alpha,
            "beta": rl_hat.beta,
            "eps": rl_hat.eps,
            **rl_stats,
            "probe_active_mean": np.nan,
            "p_lam_actor_gt_0.5": np.nan,
            "n_confirm": np.nan,
            "n_reject": np.nan,
            "n_switch_in": np.nan,
            "degenerate_probe": np.nan,
            "mcmc_accept_rate": np.nan,
        })

master_probe = pd.concat(all_rows, ignore_index=True)
master_probe_out = MASTER_PROBE_CSV
master_probe.to_csv(master_probe_out, index=False, encoding="utf-8")
print("Saved master+PROBE regressors:", master_probe_out)

fit_df = pd.DataFrame(fit_rows)
fit_out = MODEL_OUT / "model_fit_summary.csv"
fit_df.to_csv(fit_out, index=False, encoding="utf-8")
print("Saved fit summary:", fit_out)

print(f"\n[{_ts()}]Model fitting complete ({_elapsed(_cell_t0)})")
print(fit_df[["block","model","n","ll","aic","bic"]].to_string(index=False))

# QC: lambda0 variance + event consistency (confirm/reject only when probe active)
for block_qc in ["C", "S"]:
    dqc = master_probe[master_probe["block"] == block_qc].copy()
    if dqc.empty:
        continue

    if "lambda0_post" in dqc.columns:
        lam0 = pd.to_numeric(dqc["lambda0_post"], errors="coerce")
        print(f"[QC:{block_qc}] lambda0_post var={lam0.var(skipna=True):.6g}, min={lam0.min(skipna=True):.6g}, max={lam0.max(skipna=True):.6g}")

    if {"confirm", "reject", "probe_active_post"}.issubset(dqc.columns):
        bad_confirm = int(((dqc["confirm"] == 1) & (dqc["probe_active_post"] != 1)).sum())
        bad_reject = int(((dqc["reject"] == 1) & (dqc["probe_active_post"] != 1)).sum())
        print(f"[QC:{block_qc}] confirm outside probe_active: {bad_confirm}")
        print(f"[QC:{block_qc}] reject outside probe_active:  {bad_reject}")


# Attach regressors to epochs (time-based match) -> epochs_with_master_*
MAX_DT_S = 0.20


# Fallback for out-of-order notebook execution.
if "save_epochs_robust" not in globals():
    from pathlib import Path as _Path

    def save_epochs_robust(epochs_obj, out_path):
        out_path = _Path(out_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        epochs_obj.save(str(out_path), overwrite=True)
        return out_path


def attach_to_epochs(epochs_path: Path, block_label: str, out_path: Path):
    epochs = mne.read_epochs(str(epochs_path), preload=False, verbose="ERROR")
    md = epochs.metadata.copy() if epochs.metadata is not None else pd.DataFrame(index=np.arange(len(epochs)))

    if "run" not in md.columns:
        if "run_idx" in md.columns:
            md["run"] = pd.to_numeric(md["run_idx"], errors="coerce")
        else:
            raise RuntimeError(f"{block_label}: no run/run_idx in epochs metadata")

    md["onset_meg"] = epochs.events[:, 0] / epochs.info["sfreq"]
    md["run"] = pd.to_numeric(md["run"], errors="coerce")

    dfb = master_probe[master_probe["block"] == block_label].copy()
    if dfb.empty:
        raise RuntimeError(f"{block_label}: no rows in master_probe")

    keep_cols = [
        "block",
        "run",
        "onset_s",
        "stim_onset_s",
        "trial",
        "trial_x",
        "meta_row",
        "Q_actor_post",
        "q_actor_post",
        "gamma_actor_post",
        "lambda0_post",
        "lambda_actor_post",
        "lambda_alt1_post",
        "lambda_alt2_post",
        "rpe_post",
        "probe_active_post",
        "lambda0",
        "q_actor",
        "gamma_actor",
        "lambda_actor",
        "lambda_alt1",
        "lambda_alt2",
        "switch_in",
        "switch",
        "reject",
        "confirm",
        "exploration",
        "reward",
        "rew",
        "cor",
        "correct",
        "outcome",
    ]
    keep_cols = [c for c in keep_cols if c in dfb.columns]
    dfb = dfb[keep_cols].copy()

    def _merge_exact_by_trial(md_local: pd.DataFrame, dfb_local: pd.DataFrame):
        if "trial" not in md_local.columns:
            return None, None

        key_col = None
        if "trial_x" in dfb_local.columns:
            key_col = "trial_x"
        elif "meta_row" in dfb_local.columns:
            key_col = "meta_row"
        elif "trial" in dfb_local.columns:
            key_col = "trial"
        if key_col is None:
            return None, None

        left = md_local.copy().reset_index(drop=True)
        left["__row"] = np.arange(len(left))
        left["run"] = pd.to_numeric(left["run"], errors="coerce")
        left["trial_key"] = pd.to_numeric(left["trial"], errors="coerce")

        right = dfb_local.copy()
        right["run"] = pd.to_numeric(right["run"], errors="coerce")
        right["trial_key"] = pd.to_numeric(right[key_col], errors="coerce")

        left_valid = left.dropna(subset=["run", "trial_key"]).copy()
        right_valid = right.dropna(subset=["run", "trial_key"]).copy()
        if left_valid.empty or right_valid.empty:
            return None, key_col

        left_valid["run"] = left_valid["run"].astype(int)
        right_valid["run"] = right_valid["run"].astype(int)
        left_valid["trial_key"] = left_valid["trial_key"].astype(int)
        right_valid["trial_key"] = right_valid["trial_key"].astype(int)

        right_valid = right_valid.sort_values(["run", "trial_key"]).drop_duplicates(["run", "trial_key"], keep="first")

        payload_cols = [
            c for c in right_valid.columns
            if c not in ("run", "trial_key", "block", key_col) and c not in left.columns
        ]

        joined = left_valid.merge(
            right_valid[["run", "trial_key"] + payload_cols],
            on=["run", "trial_key"],
            how="left",
        )

        out = left.drop(columns=["__row"]).copy()
        for c in payload_cols:
            out[c] = np.nan
            out.loc[joined["__row"].to_numpy(), c] = joined[c].to_numpy()
        return out, key_col

    def _merge_asof(md_local: pd.DataFrame, dfb_local: pd.DataFrame):
        left = md_local.copy().reset_index(drop=True)
        left["__row"] = np.arange(len(left))
        left["run"] = pd.to_numeric(left["run"], errors="coerce")
        left["onset_meg"] = pd.to_numeric(left["onset_meg"], errors="coerce")

        right = dfb_local.copy()
        right["run"] = pd.to_numeric(right["run"], errors="coerce")
        if "onset_s" not in right.columns:
            return left.drop(columns=["__row"])
        right["onset_s"] = pd.to_numeric(right["onset_s"], errors="coerce")

        left_valid = left.dropna(subset=["run", "onset_meg"]).copy()
        right_valid = right.dropna(subset=["run", "onset_s"]).copy()
        if left_valid.empty or right_valid.empty:
            return left.drop(columns=["__row"])

        left_valid["run"] = left_valid["run"].astype(int)
        right_valid["run"] = right_valid["run"].astype(int)

        payload_cols = [
            c for c in right_valid.columns
            if c not in ("run", "onset_s", "block") and c not in left.columns
        ]

        merged_valid = pd.merge_asof(
            left_valid.sort_values(["onset_meg", "run"]),
            right_valid[["run", "onset_s"] + payload_cols].sort_values(["onset_s", "run"]),
            left_on="onset_meg",
            right_on="onset_s",
            by=["run"],
            direction="nearest",
            tolerance=MAX_DT_S,
        )

        out = left.drop(columns=["__row"]).copy()
        for c in payload_cols:
            out[c] = np.nan
            out.loc[merged_valid["__row"].to_numpy(), c] = merged_valid[c].to_numpy()
        return out

    merged_exact, key_used = _merge_exact_by_trial(md, dfb)
    merged_asof = _merge_asof(md, dfb)

    exact_missing = None
    asof_missing = None
    if merged_exact is not None and "lambda_actor" in merged_exact.columns:
        exact_missing = int(merged_exact["lambda_actor"].isna().sum())
    if "lambda_actor" in merged_asof.columns:
        asof_missing = int(merged_asof["lambda_actor"].isna().sum())

    if merged_exact is not None and (asof_missing is None or exact_missing <= asof_missing):
        merged = merged_exact
        print(f"[{block_label}] merge method: exact by run+{key_used}")
    else:
        merged = merged_asof
        print(f"[{block_label}] merge method: asof (run+time)")

    if "lambda_actor" in merged.columns:
        missing = int(merged["lambda_actor"].isna().sum())
        print(f"[{block_label}] missing regressors after merge: {missing} / {len(merged)}")

    epochs.metadata = merged
    saved_path = save_epochs_robust(epochs, out_path)
    print(f"[{block_label}] Saved: {saved_path}")



print(f"\n[{_ts()}] Attaching regressors to epochs...", flush=True)
attach_to_epochs(EPO_RUN_S, "S", EPO_MASTER_S)
attach_to_epochs(EPO_RUN_C, "C", EPO_MASTER_C)
print(f"[{_ts()}] Done — all epochs saved.")


In [ ]:
import mne

epochs_C = mne.read_epochs(str(EPO_MASTER_C), preload=False, verbose="ERROR")
epochs_S = mne.read_epochs(str(EPO_MASTER_S), preload=False, verbose="ERROR")

print("C has:", [c for c in ["Q_actor_post","gamma_actor_post","lambda0_post","lambda_actor_post","switch_in","reject","confirm"] if c in epochs_C.metadata.columns])
print("S has:", [c for c in ["Q_actor_post","gamma_actor_post","lambda0_post","lambda_actor_post","switch_in","reject","confirm"] if c in epochs_S.metadata.columns])


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LinearRegression
from scipy import stats
import matplotlib.pyplot as plt

ROI_TC_C = (BASE / "C" / "derivatives") / "fsaverage" / "roi_tc_Destrieux_C.npz"
ROI_TC_S = (BASE / "S" / "derivatives") / "fsaverage" / "roi_tc_Destrieux_S.npz"

def load_roi_timecourses(npz_path: Path):
    pack = np.load(npz_path, allow_pickle=True)
    tc = pack["data"]                    # (n_trials_valid × n_rois × n_times)
    names = pack["label_names"].astype(str)
    times = pack["times"]
    mask = pack["trial_mask"]

    # Check for mask synchronization
    n_valid = tc.shape[0]
    n_mask = mask.sum()
    if n_mask != n_valid:
        raise RuntimeError(
            f"[FATAL] {npz_path.name}: trial_mask.sum()={n_mask} != tc.shape[0]={n_valid}\n"
            f"        Mask and data are out of sync!\n"
            f"        NPZ was created with a different mask or is corrupted."
        )

    print(f"[LOAD] {npz_path.name}: {tc.shape} (trials × ROIs × times)")
    print(f"       ROI examples: {names[:3]}")
    print(f"       time: {times[0]:.3f}..{times[-1]:.3f}  n_times={len(times)}")
    print(f"       trial_mask: ✓ {mask.sum()}/{len(mask)} (SYNCHRONIZED with tc)")

    return tc, times, names, mask


# DESIGN-MATRIX (X) FOR GLM
def prepare_design_matrix(epochs_obj, trial_mask, regressor_names):
    """
    Build trialwise design matrix in trial_mask space.
    Steps:
      1) verify trial_mask is compatible with epochs_obj
      2) select masked trials
      3) drop rows with NaN in any selected regressor
      4) center all regressors
      5) orthogonalize sequence if those columns are present
      6) return X_orth, local mask, and updated regressor_names (after zero-var drop)
    """
    regressor_names = list(regressor_names)  # work on a local copy
    md = epochs_obj.metadata
    if md is None:
        raise RuntimeError("epochs_obj.metadata is None")

    # CHECK trial_mask synchronization with epochs
    # trial_mask must be the same length as epochs, as it comes from the trial_mask field in NPZ
    if len(trial_mask) != len(epochs_obj):
        raise RuntimeError(
            f"[FATAL] trial_mask length ({len(trial_mask)}) != epochs length ({len(epochs_obj)})\n"
            f"        NPZ and epochs used different sets of trials!\n"
        )

    if trial_mask.sum() == 0:
        raise RuntimeError("[FATAL] trial_mask is all False — no valid trials!")

    md = _materialize_regressor_aliases(md, list(regressor_names), "X")

    md_sel = (
        md.loc[trial_mask, regressor_names]
        .apply(pd.to_numeric, errors="coerce")
    )

    ok_local = md_sel.notna().all(axis=1).to_numpy()
    X0 = md_sel.loc[ok_local, :].to_numpy(dtype=float)

    if X0.shape[0] == 0:
        raise RuntimeError("No valid rows left for design matrix after NaN filtering")

    X0 = X0 - X0.mean(axis=0, keepdims=True)

    # Drop zero-variance columns (prevents singular X'X)
    # Use std-based threshold (1e-6) to also catch near-constant columns
    # that would make the design matrix near-singular (condition number ~10^10)
    col_var = X0.var(axis=0)
    zero_var = col_var < 1e-12
    near_zero_var = X0.std(axis=0) < 1e-6
    zero_var = zero_var | near_zero_var
    if zero_var.any():
        dropped = [regressor_names[i] for i in np.where(zero_var)[0]]
        print(f"[DESIGN] ⚠️ dropping zero-variance regressors: {dropped}")
        keep_idx = np.where(~zero_var)[0]
        X0 = X0[:, keep_idx]
        regressor_names = [regressor_names[i] for i in keep_idx]

    X_orth = X0.copy()

    orth_cols = [c for c in ORTHOGONALIZE_SEQUENCE if c in regressor_names]
    orth_idx = [regressor_names.index(c) for c in orth_cols]

    for pos, col_idx in enumerate(orth_idx):
        if pos == 0:
            continue
        prev_idx = orth_idx[:pos]
        prev = X_orth[:, prev_idx]
        y = X_orth[:, col_idx]

        G = prev.T @ prev
        if np.linalg.matrix_rank(G) == 0:
            continue

        beta = np.linalg.pinv(G) @ (prev.T @ y)
        X_orth[:, col_idx] = y - prev @ beta

    n_all = len(md)
    n_mask = int(trial_mask.sum())
    n_kept = int(ok_local.sum())
    print(
        f"[DESIGN] X shape: {X_orth.shape} "
        f"| kept {n_kept}/{n_mask} masked trials (out of {n_all} total epochs) "
        f"| orth cols: {orth_cols}"
    )

    return X_orth, ok_local, regressor_names

# GLM for one ROI
def run_glm_timecourse(roi_data, design_matrix, roi_name="", debug=False):
    """
    roi_data:       (n_trials, n_times)     — averaged among ROI time course.
    design_matrix:  (n_trials, n_regressors) — X with no intercept.
    Gives:
      betas: (n_regressors × n_times)
      tvals: (n_regressors × n_times)
    """
    roi_data = np.asarray(roi_data, float)
    design_matrix = np.asarray(design_matrix, float)

    n_trials, n_times = roi_data.shape
    n_reg = design_matrix.shape[1]

    betas = np.full((n_reg, n_times), np.nan)
    tvals = np.full((n_reg, n_times), np.nan)

    for t in range(n_times):
        y = roi_data[:, t]

        if np.all(np.isnan(y)) or np.nanvar(y) == 0:
            continue

        valid = ~np.isnan(y)
        Xv = design_matrix[valid]
        yv = y[valid]

        if len(yv) < n_reg + 2:
            continue

        reg = LinearRegression(fit_intercept=True)
        reg.fit(Xv, yv)
        y_hat = reg.predict(Xv)
        resid = yv - y_hat

        n = len(yv)
        p = n_reg + 1  # + intercept
        rss = np.sum(resid ** 2)
        sigma2 = rss / max(n - p, 1)

        Xw = np.column_stack([np.ones(n), Xv])  # add intercept
        try:
            XtX_inv = np.linalg.inv(Xw.T @ Xw)
        except np.linalg.LinAlgError:
            continue

        se = np.sqrt(np.diag(sigma2 * XtX_inv)[1:])  # exclude intercept

        betas[:, t] = reg.coef_
        tvals[:, t] = reg.coef_ / se

    if debug:
        print(f"[{roi_name}] GLM ok: β-shape {betas.shape}")

    return betas, tvals

def analyze_time_windows(
    betas,
    tvals,
    times,
    regressor_names,
    windows=[(0.12, 0.30), (0.30, 0.60)],
    roi_name="",
):
    betas = np.asarray(betas, float)
    tvals = np.asarray(tvals, float)
    times = np.asarray(times, float)

    results = {}

    for w0, w1 in windows:
        wname = f"{w0:.2f}-{w1:.2f}s"
        mask = (times >= w0) & (times <= w1)

        if not np.any(mask):
            continue

        idx = np.where(mask)[0]

        mean_b = np.nanmean(betas[:, idx], axis=1)
        mean_t = np.nanmean(tvals[:, idx], axis=1)

        peak_t = np.full(len(regressor_names), np.nan)
        peak_time = np.full(len(regressor_names), np.nan)

        for r in range(len(regressor_names)):
            t_row = tvals[r, idx]
            if np.all(np.isnan(t_row)):
                continue
            k = np.nanargmax(np.abs(t_row))
            peak_t[r] = t_row[k]
            peak_time[r] = times[idx[k]]

        results[wname] = {
            "mean_betas": mean_b,
            "mean_t_stats": mean_t,
            "peak_t_stats": peak_t,
            "peak_times": peak_time,
        }

    return results

print("Loading ROI time courses...")
tc_C, times_C, roi_names_C, mask_C = load_roi_timecourses(ROI_TC_C)
tc_S, times_S, roi_names_S, mask_S = load_roi_timecourses(ROI_TC_S)

# Select regressors based on epoch lock (stimulus vs feedback)
EPOCH_LOCK = "feedback"  # "stim" or "feedback"

REGRESSORS_PRE = [
    "lambda_actor_pre",
    "lambda_alt1_pre",
    "lambda_alt2_pre",
    "probe_active_pre",
    "switch_in",
]

# Q -> gamma -> lambda0 -> lambda_actor -> lambda_alt1 -> lambda_alt2
REGRESSORS_POST = [
    "Q_actor_post",
    "gamma_actor_post",
    "lambda0_post",
    "lambda_actor_post",
    "lambda_alt1_post",
    "lambda_alt2_post",
    "switch_in",
    "reject",
    "confirm",
]

ORTHOGONALIZE_SEQUENCE = [
    "reward",  # outcome valence (positive/negative)
    "Q_actor_post",
    "gamma_actor_post",
    "lambda0_post",
    "lambda_actor_post",
    "lambda_alt1_post",
    "lambda_alt2_post",
]

REGRESSOR_ALIASES = {
    "Q_actor_post": ["Q_actor", "q_actor"],
    "gamma_actor_post": ["gamma_actor", "gamma"],
    "lambda0_post": ["lambda_0", "lambda0"],
    "lambda_actor_post": ["lambda_actor"],
    "lambda_alt1_post": ["lambda_alt1"],
    "lambda_alt2_post": ["lambda_alt2"],
    "switch_in": ["switch"],
    "lambda_actor_pre": ["lambda_actor"],
    "lambda_alt1_pre": ["lambda_alt1"],
    "lambda_alt2_pre": ["lambda_alt2"],
    "probe_active_pre": ["probe_active"],
    "reward": ["rew", "cor", "correct"],
}


def _materialize_regressor_aliases(md_df, regressors, tag):
    """Create canonical regressor columns from aliases if needed.
    If a regressor cannot be resolved from aliases, create it as zeros to keep
    a fixed design width across subjects.
    """
    md_out = md_df.copy()
    for reg in regressors:
        if reg in md_out.columns:
            continue

        resolved = False
        for alt in REGRESSOR_ALIASES.get(reg, []):
            if alt in md_out.columns:
                md_out[reg] = pd.to_numeric(md_out[alt], errors="coerce")
                print(f"[DESIGN:{tag}] alias {reg} <- {alt}")
                resolved = True
                break

        if not resolved:
            md_out[reg] = 0.0
            print(f"[DESIGN:{tag}] fill missing regressor with 0: {reg}")

    return md_out

regressor_names = REGRESSORS_POST if EPOCH_LOCK == "feedback" else REGRESSORS_PRE
print(f"[DESIGN] EPOCH_LOCK={EPOCH_LOCK} -> {regressor_names}")
print(f"[DESIGN] Orth sequence: {ORTHOGONALIZE_SEQUENCE}")


def _configure_regressors_jointly(epochs_C, epochs_S, trial_mask_C, trial_mask_S, regressors, orth_seq):
    """
    Configure regressors by examining BOTH C and S blocks together.
    """
    regs = list(regressors)
    orth = list(orth_seq)

    # resolve aliases for both blocks
    md_C = epochs_C.metadata
    md_S = epochs_S.metadata

    if md_C is None or md_S is None:
        return regs, orth

    md_C_mask = _materialize_regressor_aliases(md_C.loc[trial_mask_C].copy(), regs + orth + ["reward"], "C+S:C")
    md_S_mask = _materialize_regressor_aliases(md_S.loc[trial_mask_S].copy(), regs + orth + ["reward"], "C+S:S")

    orth = [r for r in orth if r in regs]

    # Add outcome/valence control if needed
    if "reward" not in regs:
        regs = ["reward"] + regs
        print(f"[DESIGN] fixed outcome regressor: reward")

    if "lambda0_post" in regs:
        v_C = float(pd.to_numeric(md_C_mask["lambda0_post"], errors="coerce").var(skipna=True))
        v_S = float(pd.to_numeric(md_S_mask["lambda0_post"], errors="coerce").var(skipna=True))

        print(f"[QC] lambda0_post variance: C={v_C:.6g}, S={v_S:.6g}")

        is_const_C = np.isfinite(v_C) and v_C < 1e-8
        is_const_S = np.isfinite(v_S) and v_S < 1e-8

        if is_const_C and is_const_S:
            regs = [r for r in regs if r != "lambda0_post"]
            orth = [r for r in orth if r != "lambda0_post"]
            print(f"[QC] dropped lambda0_post (near-constant in BOTH C and S)")
        elif is_const_C or is_const_S:
            block_const = ("C" if is_const_C else "") + ("S" if is_const_S else "")
            print(f"[QC] lambda0_post near-constant only in {block_const} - KEEPING in both for consistency")

    if {"confirm", "reject", "probe_active_post"}.issubset(md_C_mask.columns):
        q_C = md_C_mask[["confirm", "reject", "probe_active_post"]].apply(pd.to_numeric, errors="coerce")
        q_S = md_S_mask[["confirm", "reject", "probe_active_post"]].apply(pd.to_numeric, errors="coerce")

        bad_confirm_C = int(((q_C["confirm"] == 1) & (q_C["probe_active_post"] != 1)).sum())
        bad_reject_C = int(((q_C["reject"] == 1) & (q_C["probe_active_post"] != 1)).sum())
        bad_confirm_S = int(((q_S["confirm"] == 1) & (q_S["probe_active_post"] != 1)).sum())
        bad_reject_S = int(((q_S["reject"] == 1) & (q_S["probe_active_post"] != 1)).sum())

        print(f"[QC] confirm outside probe_active_post: C={bad_confirm_C}, S={bad_confirm_S}")
        print(f"[QC] reject outside probe_active_post:  C={bad_reject_C}, S={bad_reject_S}")

    return regs, orth


regressor_names, ORTHOGONALIZE_SEQUENCE = _configure_regressors_jointly(
    epochs_C, epochs_S,
    mask_C, mask_S,
    regressor_names,
    ORTHOGONALIZE_SEQUENCE
)

print(f"\n[DESIGN] Final regressors (from joint C/S analysis): {regressor_names}")
print(f"[DESIGN] Final orth sequence: {ORTHOGONALIZE_SEQUENCE}")


print("\n[DESIGN] preparing design for C...")
design_C, ok_C, regressor_names_C = prepare_design_matrix(epochs_C, mask_C, regressor_names)

print("[DESIGN] preparing design for S...")
design_S, ok_S, regressor_names_S = prepare_design_matrix(epochs_S, mask_S, regressor_names)

if regressor_names_C != regressor_names_S:
    print(f"[DESIGN] regressor lists differ between C and S after zero-var drop:")
    print(f"         C: {regressor_names_C}")
    print(f"         S: {regressor_names_S}")

# epochs → mask (from NPZ) → tc (valid) → ok (without NaN) → design (regressors)

# 1) tc and mask must be synchronized (checked during loading, but let's verify again)
assert mask_C.sum() == tc_C.shape[0], f"[C] mask_C.sum()={mask_C.sum()} != tc_C.shape[0]={tc_C.shape[0]}"
assert mask_S.sum() == tc_S.shape[0], f"[S] mask_S.sum()={mask_S.sum()} != tc_S.shape[0]={tc_S.shape[0]}"

# 2) trial_mask must be the length of epochs (checked during preparation, repeating for clarity)
assert len(mask_C) == len(epochs_C), f"[C] len(mask_C)={len(mask_C)} != len(epochs_C)={len(epochs_C)}"
assert len(mask_S) == len(epochs_S), f"[S] len(mask_S)={len(mask_S)} != len(epochs_S)={len(epochs_S)}"

# 3) ok is a submask in the space of masked trials, must match the size of tc
assert ok_C.shape[0] == tc_C.shape[0], f"[C] ok_C.shape[0]={ok_C.shape[0]} != tc_C.shape[0]={tc_C.shape[0]}"
assert ok_S.shape[0] == tc_S.shape[0], f"[S] ok_S.shape[0]={ok_S.shape[0]} != tc_S.shape[0]={tc_S.shape[0]}"

# 4) Design matrix must contain only valid trials from ok
assert design_C.shape[0] == ok_C.sum(), f"[C] design_C.shape[0]={design_C.shape[0]} != ok_C.sum()={ok_C.sum()}"
assert design_S.shape[0] == ok_S.sum(), f"[S] design_S.shape[0]={design_S.shape[0]} != ok_S.sum()={ok_S.sum()}"

# 5) Ensure that there are at least some trials left
assert ok_C.sum() > 0, "[C] no valid trials after NaN filtering"
assert ok_S.sum() > 0, "[S] no valid trials after NaN filtering"

# Print hierarchy of sizes for debugging
print(f"\n[✓ SANITY CHECK] Mask hierarchy is consistent:")
print(f"  C: epochs={len(epochs_C):4d} → mask={mask_C.sum():4d} → tc={tc_C.shape[0]:4d} → ok_C={ok_C.sum():4d} → design={design_C.shape[0]:4d}  regs={len(regressor_names_C)}")
print(f"  S: epochs={len(epochs_S):4d} → mask={mask_S.sum():4d} → tc={tc_S.shape[0]:4d} → ok_S={ok_S.sum():4d} → design={design_S.shape[0]:4d}  regs={len(regressor_names_S)}")

# LIST OF ROIS FOR ANALYSIS (QC-ROI)

target_rois = {
    "superiorfrontal-lh": ["G_front_sup-lh", "S_front_sup-lh"],
    "superiorfrontal-rh": ["G_front_sup-rh", "S_front_sup-rh"],
    "insula-lh":          ["G_Ins_lg_and_S_cent_ins-lh"],
    "insula-rh":          ["G_Ins_lg_and_S_cent_ins-rh"],
    "precuneus-lh":       ["G_precuneus-lh"],
    "precuneus-rh":       ["G_precuneus-rh"],
    # vmPFC (pgACC + medial orbitofrontal)
    "vmPFC-lh": ["G_and_S_cingul-Ant-lh", "G_rectus-lh", "G_subcallosal-lh",
                 "G_and_S_subcentral-lh", "S_suborbital-lh"],
    "vmPFC-rh": ["G_and_S_cingul-Ant-rh", "G_rectus-rh", "G_subcallosal-rh",
                 "G_and_S_subcentral-rh", "S_suborbital-rh"],
    # dlPFC (middle frontal gyrus, BA9/46)
    "dlPFC-lh": ["G_front_middle-lh", "S_front_middle-lh"],
    "dlPFC-rh": ["G_front_middle-rh", "S_front_middle-rh"],
}

target_roi_indices = {}
for friendly, subs in target_rois.items():
    hits = [i for i, nm in enumerate(roi_names_C) if any(s in nm for s in subs)]
    if hits:
        target_roi_indices[friendly] = hits[0]
    else:
        print(f"[ROI] not found: {friendly} ({subs})")

print(f"[ROI] found {len(target_roi_indices)} / {len(target_rois)} ROIs")


# Main GLM-cycle on ROI

glm_results = {}

for roi_name, idx in target_roi_indices.items():
    print(f"\n[GLM] ROI = {roi_name}")

    # C BLOCK
    data_C = tc_C[ok_C, idx, :]  # (n_kept_C × n_times)
    assert design_C.shape[0] == data_C.shape[0], "[C] X and Y don't match trials"
    bet_C, t_C = run_glm_timecourse(data_C, design_C, f"{roi_name}_C", debug=True)
    win_C = analyze_time_windows(
        bet_C, t_C, times_C, regressor_names_C, roi_name=f"{roi_name}_C"
    )

    # S BLOCK
    data_S = tc_S[ok_S, idx, :]  # (n_kept_S × n_times)
    assert design_S.shape[0] == data_S.shape[0], "[S] X and Y don't match trials"
    bet_S, t_S = run_glm_timecourse(data_S, design_S, f"{roi_name}_S", debug=True)
    win_S = analyze_time_windows(
        bet_S, t_S, times_S, regressor_names_S, roi_name=f"{roi_name}_S"
    )


    glm_results[roi_name] = {

        "C": {"betas": bet_C, "t_stats": t_C, "windows": win_C},
        "S": {"betas": bet_S, "t_stats": t_S, "windows": win_S},

    }


print(f"\n[STATUS] GLM complete for {len(glm_results)} ROIs")

In [ ]:

# GLM with multiple comparison correction (FDR)
# Requires: tc_C, tc_S, design_C, design_S, times_C, times_S,
#           regressor_names, ok_C, ok_S, target_roi_indices

import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
import matplotlib.pyplot as plt


def compute_pvals_from_t(tvals, dof):
    """t-stats → two-sided p-values"""
    from scipy.stats import t as t_dist
    return 2 * (1 - t_dist.cdf(np.abs(tvals), dof))


def fdr_correction(pvals, alpha=0.05):
    """Benjamini-Hochberg FDR correction."""
    pvals_flat = pvals.flatten()
    sorted_idx = np.argsort(pvals_flat)
    sorted_pvals = pvals_flat[sorted_idx]
    n = len(sorted_pvals)

    threshold_idx = np.where(sorted_pvals <= alpha * np.arange(1, n + 1) / n)[0]
    threshold = sorted_pvals[threshold_idx[-1]] if len(threshold_idx) > 0 else 0.0

    reject = pvals_flat <= threshold
    return reject.reshape(pvals.shape), threshold


def glm_with_fdr(roi_data, design_matrix, roi_name="", debug=True):
    """
    GLM for one ROI with BH-FDR correction.

    roi_data:       (n_trials, n_times)
    design_matrix:  (n_trials, n_regressors)

    Returns dict: betas, tvals, pvals (uncorrected), reject (FDR mask).
    """
    from sklearn.linear_model import LinearRegression

    roi_data = np.asarray(roi_data, float)
    design_matrix = np.asarray(design_matrix, float)

    n_trials, n_times = roi_data.shape
    n_reg = design_matrix.shape[1]

    betas = np.full((n_reg, n_times), np.nan)
    tvals = np.full((n_reg, n_times), np.nan)

    # Run OLS at each timepoint
    for t in range(n_times):
        y = roi_data[:, t]

        if np.all(np.isnan(y)) or np.nanvar(y) == 0:
            continue

        valid = ~np.isnan(y)
        Xv = design_matrix[valid]
        yv = y[valid]

        if len(yv) < n_reg + 2:
            continue

        try:
            reg = LinearRegression(fit_intercept=True)
            reg.fit(Xv, yv)
            y_hat = reg.predict(Xv)
            resid = yv - y_hat

            n = len(yv)
            p = n_reg + 1
            rss = np.sum(resid ** 2)
            sigma2 = rss / max(n - p, 1)

            Xw = np.column_stack([np.ones(n), Xv])
            XtX_inv = np.linalg.inv(Xw.T @ Xw)
            se = np.sqrt(np.diag(sigma2 * XtX_inv)[1:])
            betas[:, t] = reg.coef_
            tvals[:, t] = reg.coef_ / se
        except Exception:
            continue

    dof = n - p
    pvals = compute_pvals_from_t(tvals, dof)
    reject, threshold = fdr_correction(pvals, alpha=0.05)

    if debug:
        n_sig = reject.sum()
        print(f"  [{roi_name}] GLM done: betas={betas.shape}, "
              f"significant={n_sig}/{reject.size} ({100*n_sig/reject.size:.1f}%)")

    return {
        "betas": betas,
        "tvals": tvals,
        "pvals": pvals,
        "reject": reject,
        "dof": dof,
    }


print("\n" + "="*70)
print("GLM WITH MULTIPLE COMPARISON CORRECTION (FDR)")
print("="*70 + "\n")

glm_results = {}

for roi_name, idx in target_roi_indices.items():
    print(f"[✓] {roi_name}")

    # C block
    data_C = tc_C[ok_C, idx, :]
    result_C = glm_with_fdr(data_C, design_C, roi_name=f"{roi_name}_C")

    # S block
    data_S = tc_S[ok_S, idx, :]
    result_S = glm_with_fdr(data_S, design_S, roi_name=f"{roi_name}_S")

    glm_results[roi_name] = {
        "C": result_C,
        "S": result_S,
    }

print("\n" + "="*70)
print("RESULTS")
print("="*70)

total_uncorr = 0
total_corr = 0

for roi_name in sorted(glm_results.keys()):
    print(f"\n{roi_name}:")
    for block in ["C", "S"]:
        result = glm_results[roi_name][block]
        n_uncorr = (result["pvals"] < 0.05).sum()
        n_corr = result["reject"].sum()
        n_total = result["reject"].size
        total_uncorr += n_uncorr
        total_corr += n_corr
        print(f"  {block}: uncorrected={n_uncorr:4d}, FDR={n_corr:4d} / {n_total:6d}")

print(f"\nTOTAL:")
print(f"  Uncorrected: {total_uncorr:5d} significant")
print(f"  FDR:         {total_corr:5d} significant (error rate controlled)")

print("\n[done]")
print("\nResults in 'glm_results':")
print("  glm_results[roi_name]['C']['betas']   — regression coefficients")
print("  glm_results[roi_name]['C']['tvals']   — t-statistics")
print("  glm_results[roi_name]['C']['pvals']   — uncorrected p-values")
print("  glm_results[roi_name]['C']['reject']  — FDR-significant mask (bool)")


In [ ]:
# Quick visualisation of results for the first ROI

first_roi = list(glm_results.keys())[0]
print(f"Plotting {first_roi}...\n")

# Per-block regressor name lists (may differ if zero-var regressors were dropped)
_reg_names = {"C": regressor_names_C, "S": regressor_names_S}

for block in ["C", "S"]:
    result = glm_results[first_roi][block]
    tvals = result["tvals"]
    reject = result["reject"]
    reg_names = _reg_names[block]

    times = times_C if block == "C" else times_S

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

    # Panel 1: Raw t-values
    im1 = ax1.imshow(tvals, aspect="auto", cmap="RdBu_r", vmin=-4, vmax=4)
    ax1.set_yticks(range(len(reg_names)))
    ax1.set_yticklabels(reg_names)
    ax1.set_xlabel("Time (s)")
    ax1.set_title(f"{first_roi} - {block} Block: Raw t-values")
    plt.colorbar(im1, ax=ax1)

    # Panel 2: Significant regions (with FDR correction)
    sig_overlay = np.where(reject, 1, 0)
    im2 = ax2.imshow(sig_overlay, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
    ax2.set_yticks(range(len(reg_names)))
    ax2.set_yticklabels(reg_names)
    ax2.set_xlabel("Time (s)")
    ax2.set_title(f"{first_roi} - {block} Block: Significant (FDR corrected, p<0.05)")
    plt.colorbar(im2, ax=ax2, label="Significant")

    plt.tight_layout()
    plt.show()

print("\n[done]")


In [ ]:
# Detailed visualisation: t-value timecourses by regressor and ROI

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ALPHA_SHADE = 0.18
LINEWIDTH = 1.4
T_THRESH_DISPLAY = 2.0

roi_list = list(glm_results.keys())
n_rois = len(roi_list)

# Per-block regressor lists (may differ after zero-var dropping)
_reg_names = {"C": regressor_names_C, "S": regressor_names_S}

# Union of all regressors (C order first, then any S-only extras)
all_reg_names = list(dict.fromkeys(regressor_names_C + regressor_names_S))
n_regs = len(all_reg_names)

roi_colors = plt.cm.tab10(np.linspace(0, 0.9, n_rois))
roi_color_map = {r: roi_colors[i] for i, r in enumerate(roi_list)}

# -- Fig 1: t(t) timecourses per regressor -----------------------------------
fig, axes = plt.subplots(n_regs, 2, figsize=(16, 2.2 * n_regs), sharex="col")
if n_regs == 1:
    axes = axes[np.newaxis, :]

fig.suptitle("T-value timecourses by regressor (all ROIs)  |  shaded = FDR sig.", fontsize=12, y=1.01)

for reg_name in all_reg_names:
    row_idx = all_reg_names.index(reg_name)
    for block_idx, block in enumerate(["C", "S"]):
        ax = axes[row_idx, block_idx]
        times_b = times_C if block == "C" else times_S
        reg_list = _reg_names[block]

        for roi_name in roi_list:
            res = glm_results[roi_name][block]
            if reg_name not in reg_list:
                # regressor was dropped for this block — leave axes blank
                continue
            reg_idx = reg_list.index(reg_name)
            tvals_row = res["tvals"][reg_idx, :]
            reject_row = res["reject"][reg_idx, :]
            col = roi_color_map[roi_name]
            ax.plot(times_b, tvals_row, color=col, lw=LINEWIDTH)
            sig_segs = np.diff(np.concatenate([[0], reject_row.astype(int), [0]]))
            starts = np.where(sig_segs == 1)[0]
            ends   = np.where(sig_segs == -1)[0]
            for s, e in zip(starts, ends):
                ax.axvspan(times_b[s], times_b[min(e, len(times_b) - 1)],
                           color=col, alpha=ALPHA_SHADE, linewidth=0)

        ax.axhline(0, color="k", lw=0.6, ls="--")
        ax.axhline( T_THRESH_DISPLAY, color="gray", lw=0.5, ls=":")
        ax.axhline(-T_THRESH_DISPLAY, color="gray", lw=0.5, ls=":")
        ax.axvline(0, color="k", lw=0.7, alpha=0.4)
        if reg_name not in reg_list:
            ax.text(0.5, 0.5, "n/a (dropped)", transform=ax.transAxes,
                    ha="center", va="center", color="gray", fontsize=8)
        if row_idx == 0:
            ax.set_title("Block " + block, fontsize=11, fontweight="bold")
        if block_idx == 0:
            ax.set_ylabel(reg_name, fontsize=8.5, rotation=0, ha="right", va="center")
        if row_idx == n_regs - 1:
            ax.set_xlabel("Time (s)")
        ax.set_xlim(times_b[0], times_b[-1])

legend_handles = [Patch(color=roi_color_map[r], label=r) for r in roi_list]
fig.legend(handles=legend_handles, loc="upper right", bbox_to_anchor=(1.14, 0.98),
           fontsize=9, title="ROI")
plt.tight_layout()
plt.show()

# Fig 2: Heatmap % sig timepoints
newline = "\n"
block_labels = ["C", "S"]
heatmap_data = np.zeros((n_regs, n_rois * 2))
col_labels = []
for r_idx, roi_name in enumerate(roi_list):
    for b_idx, block in enumerate(block_labels):
        col_idx = r_idx * 2 + b_idx
        rej = glm_results[roi_name][block]["reject"]
        reg_list = _reg_names[block]
        for row_idx, reg_name in enumerate(all_reg_names):
            if reg_name in reg_list:
                reg_idx = reg_list.index(reg_name)
                heatmap_data[row_idx, col_idx] = rej[reg_idx, :].mean() * 100
            # else stays 0
        roi_short = roi_name.replace("-", newline)
        col_labels.append(roi_short + newline + block)

fig2, ax = plt.subplots(figsize=(max(10, n_rois * 2 * 1.1), n_regs * 0.55 + 2))
im = ax.imshow(heatmap_data, aspect="auto", cmap="YlOrRd", vmin=0, vmax=100)
ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=7, rotation=45, ha="right")
ax.set_yticks(range(n_regs))
ax.set_yticklabels(all_reg_names, fontsize=9)
ax.set_title("% sig. timepoints (FDR)  regressor x ROI x block", fontsize=11)
plt.colorbar(im, ax=ax, label="% sig. timepoints")
for i in range(n_regs):
    for j in range(len(col_labels)):
        val = heatmap_data[i, j]
        ax.text(j, i, f"{val:.0f}", ha="center", va="center",
                fontsize=6.5, color="black" if val < 50 else "white")
plt.tight_layout()
plt.show()

# Fig 3: C vs S scatter (only regressors present in both blocks)
common_regs = [r for r in all_reg_names if r in regressor_names_C and r in regressor_names_S]
ncols = min(4, len(common_regs))
nrows_s = (len(common_regs) + ncols - 1) // ncols if common_regs else 1
fig3, axes3 = plt.subplots(nrows_s, ncols, figsize=(ncols * 3.5, nrows_s * 3.2))
axes3_flat = np.array(axes3).flatten()

for plot_idx, reg_name in enumerate(common_regs):
    ax = axes3_flat[plot_idx]
    c_idx = regressor_names_C.index(reg_name)
    s_idx = regressor_names_S.index(reg_name)
    for roi_name in roi_list:
        t_C_row = glm_results[roi_name]["C"]["tvals"][c_idx, :]
        t_S_row = glm_results[roi_name]["S"]["tvals"][s_idx, :]
        ax.scatter(t_C_row, t_S_row, s=3, color=roi_color_map[roi_name], alpha=0.3)
    lim = max(np.nanmax(np.abs(ax.get_xlim())), np.nanmax(np.abs(ax.get_ylim())))
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.axhline(0, color="k", lw=0.5, ls="--")
    ax.axvline(0, color="k", lw=0.5, ls="--")
    ax.plot([-lim, lim], [-lim, lim], color="gray", lw=0.8, ls=":")
    ax.set_xlabel("t (C)", fontsize=8); ax.set_ylabel("t (S)", fontsize=8)
    ax.set_title(reg_name, fontsize=9)

for ax in axes3_flat[len(common_regs):]:
    ax.set_visible(False)

legend_handles2 = [Patch(color=roi_color_map[r], label=r) for r in roi_list]
fig3.legend(handles=legend_handles2, loc="lower right", fontsize=8, title="ROI")
fig3.suptitle("C vs S t-values per timepoint  (diagonal = C==S)\n(only regressors present in both blocks)", fontsize=11)
plt.tight_layout()
plt.show()

print("All plots done.")


In [ ]:
# Subject QC diagnostics — compact report

import numpy as np
import pandas as pd
from scipy import stats

SEP = "─" * 65

# Per-block regressor lists (may differ after zero-var dropping)
_reg_names = {"C": regressor_names_C, "S": regressor_names_S}

#  DATA SIZE
print("1. DATA SIZE")
print(SEP)
print(f"  SUBJECT         : {SUBJECT}")
print(f"  epochs_C        : {len(epochs_C)} trials")
print(f"  epochs_S        : {len(epochs_S)} trials")
print(f"  mask_C (valid)  : {mask_C.sum()} / {len(mask_C)}"
      f"  ({100*mask_C.sum()/len(mask_C):.1f}%)")
print(f"  mask_S (valid)  : {mask_S.sum()} / {len(mask_S)}"
      f"  ({100*mask_S.sum()/len(mask_S):.1f}%)")
print(f"  ok_C (no-NaN X) : {ok_C.sum()} / {mask_C.sum()}"
      f"  ({100*ok_C.sum()/max(mask_C.sum(),1):.1f}%)")
print(f"  ok_S (no-NaN X) : {ok_S.sum()} / {mask_S.sum()}"
      f"  ({100*ok_S.sum()/max(mask_S.sum(),1):.1f}%)")
print(f"  tc_C shape      : {tc_C.shape}  (trials × ROIs × times)")
print(f"  tc_S shape      : {tc_S.shape}")
print(f"  times_C         : {times_C[0]:.3f} .. {times_C[-1]:.3f} s  ({len(times_C)} pts)")
print(f"  times_S         : {times_S[0]:.3f} .. {times_S[-1]:.3f} s  ({len(times_S)} pts)")
print(f"  regressor_names_C : {regressor_names_C}")
print(f"  regressor_names_S : {regressor_names_S}")
print(f"  EPOCH_LOCK      : {EPOCH_LOCK}")

#  2. BEHAVIOUR
print()
print(SEP)
print("2. BEHAVIOUR")
print(SEP)
md_c = epochs_C.metadata
md_s = epochs_S.metadata
for lbl, md in [("C", md_c), ("S", md_s)]:
    if md is None:
        print(f"  [{lbl}] metadata = None  ← PROBLEM")
        continue
    rew_col = next((c for c in ["rew","cor","correct","reward"] if c in md.columns), None)
    if rew_col:
        sub = md[rew_col].loc[mask_C if lbl=="C" else mask_S]
        acc = pd.to_numeric(sub, errors="coerce").eq(1).mean()
        print(f"  [{lbl}] accuracy={acc:.3f}  (col={rew_col})")
    else:
        print(f"  [{lbl}] reward column not found")
    if "switch_in" in md.columns:
        sw = pd.to_numeric(md["switch_in"], errors="coerce")
        print(f"  [{lbl}] switch_in rate={sw.mean():.3f}  (n_switch={int(sw.sum())})")
    if "probe_active_post" in md.columns:
        pa = pd.to_numeric(md["probe_active_post"], errors="coerce")
        print(f"  [{lbl}] probe_active rate={pa.mean():.3f}")
    blk_regs = _reg_names[lbl]
    nan_regs = {r: md[r].isna().sum() for r in blk_regs if r in md.columns}
    worst = max(nan_regs, key=nan_regs.get) if nan_regs else None
    if worst:
        print(f"  [{lbl}] NaN in regressors: max={nan_regs[worst]} in '{worst}'")

# 3. MODEL FITS
print()
print(SEP)
print("3. MODEL FITS")
print(SEP)
print(f"  FIT_METHOD = {FIT_METHOD}  |  FIT_TAU_MODE = {FIT_TAU_MODE}")
print(f"  {'':20s}  {'C':>12s}  {'S':>12s}  {'combined':>12s}")
print(f"  {'LL (PROBE)':20s}  {ll_c:>12.2f}  {ll_s:>12.2f}  {ll_combined:>12.2f}")
print(f"  {'LL (RL)':20s}  {float(rl_ll_c):>12.2f}  {float(rl_ll_s):>12.2f}  {float(rl_ll):>12.2f}")
print(f"  {'PROBE beats RL':20s}  {ll_c > float(rl_ll_c):>12}  {ll_s > float(rl_ll_s):>12}")

if "fit_df" in globals():
    _f = fit_df.copy()
    for mdl in ["PROBE", "RL"]:
        row = _f[_f["model"] == mdl][["block","ll","aic","bic"]].drop_duplicates()
        if not row.empty:
            print(f"\n  {mdl}:")
            print(row.to_string(index=False))

print(f"\n  PROBE params (C): tau={p_hat_c.tau:.4f}  eta={p_hat_c.eta:.4f}"
      f"  theta={p_hat_c.theta:.4f}  beta={p_hat_c.beta:.4f}  eps={p_hat_c.eps:.4f}")
print(f"  PROBE params (S): tau={p_hat_s.tau:.4f}  eta={p_hat_s.eta:.4f}"
      f"  theta={p_hat_s.theta:.4f}  beta={p_hat_s.beta:.4f}  eps={p_hat_s.eps:.4f}")
print(f"  RL params (C)   : alpha={rl_hat_c.alpha:.4f}  beta={rl_hat_c.beta:.4f}  eps={rl_hat_c.eps:.4f}")
print(f"  RL params (S)   : alpha={rl_hat_s.alpha:.4f}  beta={rl_hat_s.beta:.4f}  eps={rl_hat_s.eps:.4f}")
print(f"  k_bias          : {k_bias:.4f}")
print(f"  tau_env (C/S)   : {tau_env_c:.4f} / {tau_env_s:.4f}")

for lbl, phat in [("C", p_hat_c), ("S", p_hat_s)]:
    if phat.eps > 0.40:
        print(f" [{lbl}] eps={phat.eps:.3f} — high noise, nearly random choices")
    if phat.beta < 0.5:
        print(f"[{lbl}] beta={phat.beta:.3f} — low inv-temperature, unstable decisions")

# 4. GLM SUMMARY
print()
print(SEP)
print("4. GLM SUMMARY (FDR α=0.05)")
print(SEP)
print(f"  {'ROI':<22s} {'blk':>4s} {'reg':<22s}  {'%FDR':>6s}  {'peak|t|':>8s}  {'peak_t(s)':>10s}")
print(f"  {'-'*22} {'----':>4} {'-'*22}  {'------':>6}  {'--------':>8}  {'----------':>10}")
for roi_name in sorted(glm_results.keys()):
    for blk in ["C", "S"]:
        res = glm_results[roi_name][blk]
        times_b = times_C if blk == "C" else times_S
        blk_regs = _reg_names[blk]
        for ri, rn in enumerate(blk_regs):
            tr = res["tvals"][ri]
            rr = res["reject"][ri]
            pct = 100 * rr.mean()
            if np.all(np.isnan(tr)):
                continue
            pk_idx = np.nanargmax(np.abs(tr))
            pk_t = float(tr[pk_idx])
            pk_time = float(times_b[pk_idx])
            flag = " ★" if pct > 10 else ""
            print(f"  {roi_name:<22s} {blk:>4s} {rn:<22s}  {pct:>5.1f}%  {abs(pk_t):>8.2f}  {pk_time:>10.3f}{flag}")

# 5. QC FLAGS
print()
print(SEP)
print("5. QC FLAGS")
print(SEP)
flags = []

for lbl, n in [("C", ok_C.sum()), ("S", ok_S.sum())]:
    if n < 100:
        flags.append(f" [{lbl}] low trial count for GLM: n={n} (< 100)")

for roi_name in glm_results:
    for blk in ["C","S"]:
        res = glm_results[roi_name][blk]
        blk_regs = _reg_names[blk]
        if "lambda0_post" in blk_regs:
            ri = blk_regs.index("lambda0_post")
            pct = 100 * res["reject"][ri].mean()
            if pct > 80:
                flags.append(f" [{roi_name}/{blk}] lambda0_post FDR={pct:.0f}% — dominates; check multicollinearity")

for lbl, ll_p, ll_r in [("C", ll_c, float(rl_ll_c)), ("S", ll_s, float(rl_ll_s))]:
    delta = ll_p - ll_r
    if delta > 0:
        flags.append(f"✓  [{lbl}] PROBE better than RL  (ΔLL={delta:.1f})")
    else:
        flags.append(f"[{lbl}] RL better than PROBE  (ΔLL={delta:.1f})")

for lbl, n_con, n_rej in [
    ("C", int(bad_confirm), int(bad_reject)),
    ("S", int(bad_confirm), int(bad_reject)),
]:
    if n_con + n_rej > 5:
        flags.append(f"[{lbl}] confirm/reject outside probe_active: {n_con}+{n_rej} trials")

pct_C_total = sum(glm_results[r]["C"]["reject"].mean() for r in glm_results) / len(glm_results)
pct_S_total = sum(glm_results[r]["S"]["reject"].mean() for r in glm_results) / len(glm_results)
if pct_C_total < 0.005 and pct_S_total > 0.05:
    flags.append(" C block nearly silent, S block significant — block asymmetry")
elif pct_S_total < 0.005 and pct_C_total > 0.05:
    flags.append(" S block nearly silent, C block significant — block asymmetry")

if not flags:
    flags.append("✓  No obvious technical issues found")

for f in flags:
    print(f"  {f}")

print()
print(SEP)
print("DONE")
print(SEP)


In [ ]:
# Compact diagnostics dump (machine-readable format)
import numpy as np, sys

_reg_names = {"C": regressor_names_C, "S": regressor_names_S}

lines = []
lines.append(f"SUBJECT={SUBJECT}")
lines.append(f"epochs C={len(epochs_C)} S={len(epochs_S)}")
lines.append(f"mask   C={mask_C.sum()} S={mask_S.sum()}")
lines.append(f"ok     C={ok_C.sum()} S={ok_S.sum()}")
lines.append(f"tc     C={tc_C.shape} S={tc_S.shape}")
lines.append(f"times  C={times_C[0]:.3f}..{times_C[-1]:.3f} n={len(times_C)}")
lines.append(f"EPOCH_LOCK={EPOCH_LOCK}")
lines.append(f"regressors_C={regressor_names_C}")
lines.append(f"regressors_S={regressor_names_S}")
lines.append(f"LL PROBE C={ll_c:.2f} S={ll_s:.2f} comb={ll_combined:.2f}")
lines.append(f"LL RL    C={float(rl_ll_c):.2f} S={float(rl_ll_s):.2f}")
lines.append(f"PROBE > RL? C={ll_c > float(rl_ll_c)} S={ll_s > float(rl_ll_s)}")
lines.append(f"PROBE_C tau={p_hat_c.tau:.4f} eta={p_hat_c.eta:.4f} theta={p_hat_c.theta:.4f} beta={p_hat_c.beta:.4f} eps={p_hat_c.eps:.4f}")
lines.append(f"PROBE_S tau={p_hat_s.tau:.4f} eta={p_hat_s.eta:.4f} theta={p_hat_s.theta:.4f} beta={p_hat_s.beta:.4f} eps={p_hat_s.eps:.4f}")
lines.append(f"RL_C alpha={rl_hat_c.alpha:.4f} beta={rl_hat_c.beta:.4f} eps={rl_hat_c.eps:.4f}")
lines.append(f"RL_S alpha={rl_hat_s.alpha:.4f} beta={rl_hat_s.beta:.4f} eps={rl_hat_s.eps:.4f}")
lines.append(f"k_bias={k_bias:.4f} tau_env_C={tau_env_c:.4f} tau_env_S={tau_env_s:.4f}")

# top GLM hits (>5% FDR sig)
lines.append("--- GLM top (>5% FDR) ---")
for roi in sorted(glm_results):
    for blk in ["C", "S"]:
        res = glm_results[roi][blk]
        tb = times_C if blk == "C" else times_S
        blk_regs = _reg_names[blk]
        for ri, rn in enumerate(blk_regs):
            pct = 100 * res["reject"][ri].mean()
            if pct > 5:
                tr = res["tvals"][ri]
                pi = np.nanargmax(np.abs(tr))
                lines.append(f"  {roi}/{blk} {rn}: {pct:.1f}% peak_t={tr[pi]:.2f} @{tb[pi]:.3f}s")

# regressors with 0% FDR everywhere (common to both blocks)
lines.append("--- zero-signal regressors (0% FDR in all ROI×block) ---")
common_regs = [r for r in regressor_names_C if r in regressor_names_S]
for rn in common_regs:
    ri_C = regressor_names_C.index(rn)
    ri_S = regressor_names_S.index(rn)
    all_zero = all(
        glm_results[roi][blk]["reject"][ri_C if blk == "C" else ri_S].sum() == 0
        for roi in glm_results for blk in ["C", "S"]
    )
    if all_zero:
        lines.append(f"  {rn}: 0% everywhere")

lines.append(f"bad_confirm={bad_confirm} bad_reject={bad_reject}")

print("\n".join(lines))


In [ ]:
# Multicollinearity check: correlation matrix + VIF for design_C and design_S

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

_reg_names = {"C": regressor_names_C, "S": regressor_names_S}

def compute_vif(X, names):
    """Variance Inflation Factor for each column of X (no intercept)."""
    from sklearn.linear_model import LinearRegression
    n, p = X.shape
    vif = {}
    for i, name in enumerate(names):
        y = X[:, i]
        others = np.delete(X, i, axis=1)
        if others.shape[1] == 0:
            vif[name] = 1.0
            continue
        reg = LinearRegression(fit_intercept=True).fit(others, y)
        ss_res = np.sum((y - reg.predict(others)) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 1e-12 else 0.0
        vif[name] = 1.0 / (1 - r2) if r2 < 1.0 else np.inf
    return vif

SEP = "─" * 65

for label, X in [("C", design_C), ("S", design_S)]:
    names = _reg_names[label]
    n, p = X.shape
    print(SEP)
    print(f"BLOCK {label}  (n={n} trials, p={p} regressors)")
    print(SEP)

    # 1. Check for zero variance
    stds = X.std(axis=0)
    zero_var = stds < 1e-6
    if zero_var.any():
        print(" zero variance:", [names[i] for i in np.where(zero_var)[0]])
    else:
        print("  ✓ all regressors have non-zero variance")

    # 2. VIF
    vif = compute_vif(X, names)
    print(f"\n  VIF (Variance Inflation Factor):")
    print(f"  {'Regressor':<25s}   {'VIF':>8s}   {'Flag'}")
    for name, v in vif.items():
        flag = (" CRITICAL (>10)" if v > 10
                else " moderate (5-10)" if v > 5
                else "✓")
        print(f"  {name:<25s}   {v:>8.2f}   {flag}")

    # 3. Condition number
    _, sv, _ = np.linalg.svd(X, full_matrices=False)
    cond = sv[0] / sv[-1] if sv[-1] > 1e-12 else np.inf
    flag_cond = ("CRITICAL (>1000)" if cond > 1000
                 else "moderate (>100)" if cond > 100
                 else "✓")
    print(f"\n  Condition number of X: {cond:.1f}  {flag_cond}")
    print(f"  Singular values: {np.round(sv, 3)}")

    print()

# Correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, label, X in zip(axes, ["C", "S"], [design_C, design_S]):
    names = _reg_names[label]
    Xz = (X - X.mean(0)) / (X.std(0) + 1e-12)
    R = (Xz.T @ Xz) / len(X)
    np.fill_diagonal(R, 1.0)

    im = ax.imshow(R, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=8)
    ax.set_title(f"Correlation matrix — block {label}", fontsize=11)

    for i in range(len(names)):
        for j in range(len(names)):
            v = R[i, j]
            color = "white" if abs(v) > 0.5 else "black"
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=6.5, color=color)

plt.colorbar(im, ax=axes[-1], shrink=0.7, label="Pearson r")
plt.suptitle("Regressor correlation in design matrix\n"
             "|r|>0.5 → potential multicollinearity; |r|>0.8 → serious",
             fontsize=11)
plt.tight_layout()
plt.show()

# VIF bar chart
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
for ax, label, X in zip(axes2, ["C", "S"], [design_C, design_S]):
    names = _reg_names[label]
    vif = compute_vif(X, names)
    vif_vals = [vif[n] for n in names]
    colors = ["#d62728" if v > 10 else "#ff7f0e" if v > 5 else "#2ca02c"
              for v in vif_vals]
    bars = ax.barh(names, vif_vals, color=colors)
    ax.axvline(5, color="orange", lw=1.5, ls="--", label="VIF=5 (moderate)")
    ax.axvline(10, color="red", lw=1.5, ls="--", label="VIF=10 (critical)")
    ax.set_xlabel("VIF")
    ax.set_title(f"VIF — block {label}")
    ax.legend(fontsize=8)
    for bar, v in zip(bars, vif_vals):
        ax.text(min(v + 0.2, ax.get_xlim()[1] * 0.95), bar.get_y() + bar.get_height() / 2,
                f"{v:.1f}", va="center", fontsize=8)

plt.suptitle("Variance Inflation Factor by regressor\n"
             "VIF>5 moderate, VIF>10 critical multicollinearity", fontsize=11)
plt.tight_layout()
plt.show()

# High-correlation pairs
print(SEP)
print("Pairs with |r| > 0.3  (threshold for MEG GLM)")
print(SEP)
for label, X in [("C", design_C), ("S", design_S)]:
    names = _reg_names[label]
    Xz = (X - X.mean(0)) / (X.std(0) + 1e-12)
    R = (Xz.T @ Xz) / len(X)
    np.fill_diagonal(R, 0)
    pairs = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            if abs(R[i, j]) > 0.3:
                pairs.append((names[i], names[j], R[i, j]))
    pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    print(f"\n  Block {label}:")
    if not pairs:
        print("    ✓ No pairs with |r|>0.3")
    for a, b, r in pairs:
        sev = "🔴" if abs(r) > 0.7 else "🟡" if abs(r) > 0.5 else "⚪"
        print(f"    {sev} {a:<25s} ↔ {b:<25s}  r={r:+.3f}")


In [ ]:

# Diagnostics: why does lambda0_post dominate?
#
# Competing hypotheses:
#  H1. Low variance (signal near constant → small SE → inflated t)
#  H2. Skewed / clipped distribution (values near 0 or 1)
#  H3. Temporal autocorrelation (λ0 changes slowly → trials not i.i.d.)
#  H4. Genuine neural effect (λ0 is actually encoded in the brain)
#  H5. reward=0 everywhere — is reward absorbed into λ0?

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEP = "─" * 65
lam0_idx = regressor_names.index("lambda0_post")
rew_idx   = regressor_names.index("reward")

#  H1+H2: Distribution of lambda0_post and reward
print(SEP)
print("H1+H2: DISTRIBUTION OF lambda0_post AND reward")
print(SEP)
for lbl, X in [("C", design_C), ("S", design_S)]:
    lam = X[:, lam0_idx]
    rew = X[:, rew_idx]
    print(f"\n  Block {lbl}  (n={len(lam)}):")
    print(f"    lambda0_post: mean={lam.mean():.4f}  std={lam.std():.4f}"
          f"  min={lam.min():.4f}  max={lam.max():.4f}"
          f"  skew={stats.skew(lam):.2f}")
    print(f"    reward:       mean={rew.mean():.4f}  std={rew.std():.4f}"
          f"  min={rew.min():.4f}  max={rew.max():.4f}"
          f"  skew={stats.skew(rew):.2f}")
    # fraction of trials outside (-0.3, 0.3) after centering — high = good spread
    clip = np.mean((lam < -0.3) | (lam > 0.3)) * 100
    print(f"    lambda0_post outside [-0.3,0.3]: {clip:.1f}%  "
          f"(after centering; high = good spread)")

fig, axes = plt.subplots(2, 3, figsize=(15, 7))

for col_ax, (lbl, X) in enumerate([("C", design_C), ("S", design_S)]):
    lam = X[:, lam0_idx]
    rew = X[:, rew_idx]
    trial_idx = np.arange(len(lam))

    axes[col_ax, 0].hist(lam, bins=40, color="#2196F3", edgecolor="k", alpha=0.8)
    axes[col_ax, 0].set_title(f"[{lbl}] λ0_post distribution (centred)")
    axes[col_ax, 0].set_xlabel("value"); axes[col_ax, 0].set_ylabel("count")

    axes[col_ax, 1].hist(rew, bins=20, color="#FF9800", edgecolor="k", alpha=0.8)
    axes[col_ax, 1].set_title(f"[{lbl}] reward distribution (centred)")
    axes[col_ax, 1].set_xlabel("value")

    axes[col_ax, 2].plot(trial_idx, lam, lw=0.5, color="#2196F3", alpha=0.7)
    axes[col_ax, 2].set_title(f"[{lbl}] λ0_post vs trial order")
    axes[col_ax, 2].set_xlabel("trial #"); axes[col_ax, 2].set_ylabel("λ0 (centred)")

plt.tight_layout()
plt.show()

# H3: Autocorrelation of lambda0_post
print()
print(SEP)
print("H3: AUTOCORRELATION OF lambda0_post (lags 1..20)")
print(SEP)
for lbl, X in [("C", design_C), ("S", design_S)]:
    lam = X[:, lam0_idx]
    acf = [1.0] + [np.corrcoef(lam[:-k], lam[k:])[0, 1] for k in range(1, 21)]
    sig95 = 1.96 / np.sqrt(len(lam))
    n_sig = sum(abs(a) > sig95 for a in acf[1:])
    print(f"  Block {lbl}: lags 1..20 ACF = {[round(a,3) for a in acf[1:6]]}..."
          f"  ({n_sig}/20 lags sig, 95% threshold={sig95:.3f})")

fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4))
for ax, lbl, X in zip(axes2, ["C", "S"], [design_C, design_S]):
    lam = X[:, lam0_idx]
    acf = [1.0] + [np.corrcoef(lam[:-k], lam[k:])[0, 1] for k in range(1, 41)]
    lags = range(len(acf))
    sig95 = 1.96 / np.sqrt(len(lam))
    ax.bar(lags, acf, color=["#2196F3" if abs(a) > sig95 else "#90CAF9" for a in acf])
    ax.axhline( sig95, color="red", lw=1, ls="--", label="±95% CI")
    ax.axhline(-sig95, color="red", lw=1, ls="--")
    ax.axhline(0, color="k", lw=0.5)
    ax.set_title(f"ACF λ0_post — block {lbl}")
    ax.set_xlabel("lag (trials)"); ax.set_ylabel("autocorrelation")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# H4: Explained variance R² for lambda0 vs reward
print()
print(SEP)
print("H4: R² for lambda0_post vs reward in GLM (1 ROI, time-averaged)")
print(SEP)
roi_test = "superiorfrontal-lh"
idx_roi = target_roi_indices[roi_test]

for lbl, X, tc_arr, ok_arr, times_arr in [
    ("C", design_C, tc_C, ok_C, times_C),
    ("S", design_S, tc_S, ok_S, times_S),
]:
    Y = tc_arr[ok_arr, idx_roi, :]   # (n_trials, n_times)
    r2_lam0  = np.full(len(times_arr), np.nan)
    r2_rew   = np.full(len(times_arr), np.nan)
    r2_full  = np.full(len(times_arr), np.nan)
    r2_null  = np.full(len(times_arr), np.nan)

    for t in range(len(times_arr)):
        y = Y[:, t]
        valid = ~np.isnan(y)
        if valid.sum() < 20:
            continue
        yv = y[valid]; Xv = X[valid]
        ss_tot = np.var(yv) * len(yv)
        if ss_tot < 1e-12:
            continue

        def ols_r2(predictors):
            from sklearn.linear_model import LinearRegression
            reg = LinearRegression(fit_intercept=True).fit(predictors, yv)
            ss_res = np.sum((yv - reg.predict(predictors))**2)
            return max(0.0, 1 - ss_res / ss_tot)

        r2_full[t]  = ols_r2(Xv)
        r2_lam0[t]  = ols_r2(Xv[:, lam0_idx:lam0_idx+1])   # λ0 only
        r2_rew[t]   = ols_r2(Xv[:, rew_idx:rew_idx+1])      # reward only

    print(f"  [{lbl}] R² mean (full model): {np.nanmean(r2_full):.4f}")
    print(f"  [{lbl}] R² mean (λ0 alone):   {np.nanmean(r2_lam0):.4f}")
    print(f"  [{lbl}] R² mean (rew alone):  {np.nanmean(r2_rew):.4f}")

    fig3, ax = plt.subplots(figsize=(12, 3))
    ax.fill_between(times_arr, r2_full,  alpha=0.3, color="gray",   label="full model")
    ax.fill_between(times_arr, r2_lam0, alpha=0.6, color="#2196F3", label="λ0_post only")
    ax.fill_between(times_arr, r2_rew,  alpha=0.6, color="#FF9800", label="reward only")
    ax.axvline(0, color="k", lw=0.7, alpha=0.4)
    ax.set_title(f"R² timecourse — {roi_test}  [{lbl}]")
    ax.set_xlabel("time (s)"); ax.set_ylabel("R²")
    ax.legend(); plt.tight_layout(); plt.show()

#  H5: reward in raw metadata (before centering)
print()
print(SEP)
print("H5: reward IN RAW METADATA (before centering)")
print(SEP)
for lbl, epo, mask in [("C", epochs_C, mask_C), ("S", epochs_S, mask_S)]:
    md = epo.metadata
    rew_raw_col = next((c for c in ["rew","cor","correct","reward"] if c in md.columns), None)
    if rew_raw_col:
        vals = pd.to_numeric(md[rew_raw_col].loc[mask], errors="coerce").dropna()
        vc = vals.value_counts().sort_index()
        print(f"  [{lbl}] '{rew_raw_col}' value_counts: {dict(vc)}")
        print(f"         mean={vals.mean():.3f}  std={vals.std():.3f}  nunique={vals.nunique()}")
    else:
        print(f"  [{lbl}] reward column not found")

    rew_design = (design_C if lbl=="C" else design_S)[:, rew_idx]
    print(f"  [{lbl}] design reward: std={rew_design.std():.4f}  nunique={len(np.unique(rew_design))}")

print()
print(SEP)
print("DIAGNOSTICS SUMMARY")
print(SEP)


In [ ]:

# GLM WITH HAC (NEWEY-WEST) STANDARD ERRORS
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt


def _nw_bandwidth(n: int) -> int:
    return max(1, int(np.floor(4.0 * (n / 100.0) ** (2.0 / 9.0))))


def _hac_se(X_with_intercept, residuals, bandwidth):
    Xw = X_with_intercept
    e = residuals
    n, k = Xw.shape
    scores = Xw * e[:, None]   # (n, k)
    S = scores.T @ scores      # lag 0
    for h in range(1, bandwidth + 1):
        w = 1.0 - h / (bandwidth + 1.0)
        cross = scores[h:].T @ scores[:-h]
        S += w * (cross + cross.T)
    try:
        XtX_inv = np.linalg.inv(Xw.T @ Xw)
    except np.linalg.LinAlgError:
        return np.full(k, np.nan)
    V = XtX_inv @ S @ XtX_inv
    d = np.diag(V)
    return np.sqrt(np.where(d < 0, np.nan, d))


def glm_with_hac_fdr(roi_data, design_matrix, alpha=0.05, bandwidth_override=None):
    roi_data = np.asarray(roi_data, float)
    design_matrix = np.asarray(design_matrix, float)
    n_trials, n_times = roi_data.shape
    n_reg = design_matrix.shape[1]

    betas     = np.full((n_reg, n_times), np.nan)
    tvals_ols = np.full((n_reg, n_times), np.nan)
    tvals_hac = np.full((n_reg, n_times), np.nan)
    bw = bandwidth_override if bandwidth_override is not None else _nw_bandwidth(n_trials)

    for t in range(n_times):
        y = roi_data[:, t]
        if np.all(np.isnan(y)) or np.nanvar(y) == 0:
            continue
        valid = ~np.isnan(y)
        Xv, yv = design_matrix[valid], y[valid]
        nv = len(yv)
        if nv < n_reg + 5:
            continue
        reg = LinearRegression(fit_intercept=True).fit(Xv, yv)
        resid = yv - reg.predict(Xv)
        Xw = np.column_stack([np.ones(nv), Xv])
        rss = np.sum(resid ** 2)
        sigma2 = rss / max(nv - n_reg - 1, 1)
        try:
            XtX_inv = np.linalg.inv(Xw.T @ Xw)
            se_ols = np.sqrt(np.diag(sigma2 * XtX_inv)[1:])
        except np.linalg.LinAlgError:
            continue
        se_hac = _hac_se(Xw, resid, bw)[1:]
        betas[:, t]     = reg.coef_
        tvals_ols[:, t] = reg.coef_ / se_ols
        with np.errstate(invalid="ignore"):
            tvals_hac[:, t] = reg.coef_ / se_hac

    dof = n_trials - n_reg - 1
    from scipy.stats import t as t_dist
    def _p(tv):
        return 2 * (1 - t_dist.cdf(np.abs(tv), dof))
    def _fdr(pv):
        flat = pv.flatten()
        idx_s = np.argsort(flat)
        sp = flat[idx_s]
        m = len(sp)
        thr_idx = np.where(sp <= alpha * np.arange(1, m + 1) / m)[0]
        thr = sp[thr_idx[-1]] if len(thr_idx) > 0 else 0.0
        return (flat <= thr).reshape(pv.shape)

    return {
        "betas": betas,
        "tvals_ols": tvals_ols, "tvals_hac": tvals_hac,
        "pvals_ols": _p(tvals_ols), "pvals_hac": _p(tvals_hac),
        "reject_ols": _fdr(_p(tvals_ols)),
        "reject_hac": _fdr(_p(tvals_hac)),
        "bandwidth": bw, "dof": dof,
    }


# Run HAC GLM (rule-of-thumb bandwidth)
print(f"HAC GLM  |  rule-of-thumb bw={_nw_bandwidth(ok_C.sum())}  "
      f"(C={ok_C.sum()} trials, S={ok_S.sum()} trials)")
glm_hac = {}
for roi_name, idx in target_roi_indices.items():
    res_C = glm_with_hac_fdr(tc_C[ok_C, idx, :], design_C)
    res_S = glm_with_hac_fdr(tc_S[ok_S, idx, :], design_S)
    glm_hac[roi_name] = {"C": res_C, "S": res_S}

# Summary: OLS vs HAC-6
print()
print(f"  {'ROI':<22} {'blk':>3}  {'regressor':<22}  {'OLS%':>6}  {'HAC6%':>6}")
print(f"  {'-'*80}")
_reg_names = {"C": regressor_names_C, "S": regressor_names_S}
for roi_name in sorted(glm_hac.keys()):
    for blk in ["C", "S"]:
        for ri, rn in enumerate(_reg_names[blk]):
            p_ols = 100 * glm_results[roi_name][blk]["reject"][ri].mean()
            p_hac = 100 * glm_hac[roi_name][blk]["reject_hac"][ri].mean()
            flag = " ⚠️" if p_ols > 5 else ""
            print(f"  {roi_name:<22} {blk:>3}  {rn:<22}  {p_ols:>5.1f}%  {p_hac:>5.1f}%{flag}")

# plot
roi_plot = "superiorfrontal-lh"
fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharex="col")
fig.suptitle(f"OLS vs HAC t-values  |  {roi_plot}  |  lambda0_post", fontsize=11)
for row_ax, blk in enumerate(["C", "S"]):
    times_b = times_C if blk == "C" else times_S
    blk_regs = _reg_names[blk]
    lam0_idx_local = blk_regs.index("lambda0_post") if "lambda0_post" in blk_regs else None
    if lam0_idx_local is None:
        continue
    t_ols = glm_results[roi_plot][blk]["tvals"][lam0_idx_local]
    t_hac = glm_hac[roi_plot][blk]["tvals_hac"][lam0_idx_local]
    rej_ols = glm_results[roi_plot][blk]["reject"][lam0_idx_local]
    rej_hac = glm_hac[roi_plot][blk]["reject_hac"][lam0_idx_local]
    for col_ax, (t_arr, rej_arr, label, color) in enumerate([
        (t_ols, rej_ols, "OLS", "#1565C0"),
        (t_hac, rej_hac, f"HAC bw={glm_hac[roi_plot][blk]['bandwidth']}", "#C62828"),
    ]):
        ax = axes[row_ax, col_ax]
        ax.plot(times_b, t_arr, color=color, lw=1.3)
        segs = np.diff(np.concatenate([[0], rej_arr.astype(int), [0]]))
        for s, e in zip(np.where(segs == 1)[0], np.where(segs == -1)[0]):
            ax.axvspan(times_b[s], times_b[min(e, len(times_b)-1)], color=color, alpha=0.18, linewidth=0)
        ax.axhline(0, color="k", lw=0.5, ls="--")
        ax.axhline(2, color="gray", lw=0.5, ls=":")
        ax.axhline(-2, color="gray", lw=0.5, ls=":")
        ax.axvline(0, color="k", lw=0.7, alpha=0.4)
        ax.set_title(f"[{blk}] {label}  |  FDR={100*rej_arr.mean():.1f}%", fontsize=10)
        ax.set_xlabel("time (s)"); ax.set_ylabel("t-value")
        ax.set_xlim(times_b[0], times_b[-1])
plt.tight_layout(); plt.show()
print("\n[done] glm_hac ready. Next: HAC-20 + final summary.")


In [ ]:

# Conservative HAC (bandwidth = n_sig_lags) + final interpretation


BW_CONSERVATIVE = 20   # = n significant ACF lags from diagnostics

print(f"Conservative Newey-West bandwidth = {BW_CONSERVATIVE}")
print("(= n significant ACF lags, rho≈0.83)")
print()

glm_hac_cons = {}
for roi_name, idx in target_roi_indices.items():
    data_C = tc_C[ok_C, idx, :]
    data_S = tc_S[ok_S, idx, :]
    res_C = glm_with_hac_fdr(data_C, design_C, bandwidth_override=BW_CONSERVATIVE)
    res_S = glm_with_hac_fdr(data_S, design_S, bandwidth_override=BW_CONSERVATIVE)
    glm_hac_cons[roi_name] = {"C": res_C, "S": res_S}

# Summary: OLS vs HAC-6 vs HAC-20
print(f"{'ROI':<22} {'Blk':>3}  {'Regressor':<22}  "
      f"{'OLS%':>6}  {'HAC6%':>6}  {'HAC20%':>6}  {'VERDICT'}")
print("-" * 95)
lam_idx = regressor_names.index("lambda0_post")
rew_idx2 = regressor_names.index("reward")
show_regs = [rew_idx2, lam_idx]   # reward and lambda0_post

for roi_name in sorted(target_roi_indices.keys()):
    for blk in ["C", "S"]:
        r_ols  = glm_results[roi_name][blk]
        r_hac6 = glm_hac[roi_name][blk]
        r_hac20 = glm_hac_cons[roi_name][blk]
        for ri in show_regs:
            rn = regressor_names[ri]
            p_ols   = 100 * r_ols["reject"][ri].mean()
            p_hac6  = 100 * r_hac6["reject_hac"][ri].mean()
            p_hac20 = 100 * r_hac20["reject_hac"][ri].mean()
            # verdict
            if p_hac20 > 10:
                verdict = "✅ REAL EFFECT"
            elif p_hac20 > 2:
                verdict = "⚠️  weak effect"
            else:
                verdict = "❌ autocorr. artifact"
            print(f"{roi_name:<22} {blk:>3}  {rn:<22}  "
                  f"{p_ols:>5.1f}%  {p_hac6:>5.1f}%  {p_hac20:>5.1f}%  {verdict}")
    print()

# Final interpretation
print()
print("=" * 65)
print("INTERPRETATION")
print("=" * 65)

lam_survives = []
for roi_name in target_roi_indices:
    for blk in ["C", "S"]:
        p20 = 100 * glm_hac_cons[roi_name][blk]["reject_hac"][lam_idx].mean()
        lam_survives.append(p20 > 5)

if sum(lam_survives) > len(lam_survives) // 2:
    print("done")


In [ ]:
# NPZ EXPORT - compatible with group-level pipeline
# Shape convention: (n_roi, n_reg, n_times)

import numpy as np

# Use glm_hac_cons (bw=20, from cell 21) if available;
# fall back to glm_hac (bw≈6, rule-of-thumb) if cell 21 was skipped.
_glm_src = glm_hac_cons if 'glm_hac_cons' in vars() else glm_hac
_bw_used  = 20           if 'glm_hac_cons' in vars() else 6
print(f"[INFO] NPZ export using bandwidth={_bw_used} "
      f"({'glm_hac_cons' if _bw_used == 20 else 'glm_hac'})")

roi_list = list(_glm_src.keys())           # 10 ROIs
n_roi    = len(roi_list)
n_reg_C  = len(regressor_names_C)
n_reg_S  = len(regressor_names_S)
n_t_C    = len(times_C)
n_t_S    = len(times_S)

def _stack(block, key):
    return np.stack([_glm_src[r][block][key] for r in roi_list])

betas_C    = _stack("C", "betas")          # (n_roi, n_reg, n_t_C)
tvals_C    = _stack("C", "tvals_hac")
pvals_C    = _stack("C", "pvals_hac")
reject_C   = _stack("C", "reject_hac")

betas_S    = _stack("S", "betas")          # (n_roi, n_reg, n_t_S)
tvals_S    = _stack("S", "tvals_hac")
pvals_S    = _stack("S", "pvals_hac")
reject_S   = _stack("S", "reject_hac")

npz_out = MODEL_OUT / f"glm_hac20_{SUBJECT}.npz"
np.savez_compressed(
    npz_out,
    subject          = SUBJECT,
    roi_names        = np.array(roi_list),
    regressor_names_C = np.array(regressor_names_C),
    regressor_names_S = np.array(regressor_names_S),
    times_C          = times_C,
    times_S          = times_S,
    betas_C          = betas_C,
    tvals_hac20_C    = tvals_C,
    pvals_hac20_C    = pvals_C,
    reject_hac20_C   = reject_C,
    betas_S          = betas_S,
    tvals_hac20_S    = tvals_S,
    pvals_hac20_S    = pvals_S,
    reject_hac20_S   = reject_S,
    bandwidth        = _bw_used,
    alpha_fdr        = 0.05,
    n_trials_C       = int(ok_C.sum()),
    n_trials_S       = int(ok_S.sum()),
    epoch_lock       = EPOCH_LOCK,
    fit_tau_mode     = FIT_TAU_MODE,
)
print(f"[SAVED] {npz_out}")
print(f"  ROIs ({n_roi}): {roi_list}")
print(f"  Regressors C ({n_reg_C}): {regressor_names_C}")
print(f"  Regressors S ({n_reg_S}): {regressor_names_S}")
print(f"  Shape C: {betas_C.shape}  S: {betas_S.shape}")

# Quick sanity: how many ROI×reg×block survive HAC FDR >5%?
hits = []
for bi, blk in enumerate(["C", "S"]):
    rej = reject_C if blk == "C" else reject_S
    for ri, rn in enumerate(roi_list):
        for gi, gn in enumerate(_reg_names[blk]):
            pct = 100 * rej[ri, gi].mean()
            if pct > 5:
                hits.append(f"  {rn}/{blk} {gn}: {pct:.0f}%")
print(f"\n[HAC FDR >5%] {len(hits)} ROI×reg×block combinations:")
for h in hits:
    print(h)


In [ ]:
# Saves per-session files with window_betas_by_group + full_betas_by_group
# Path: {MEG_ROOT}/{SUBJECT}/{ses}/derivatives/fsaverage/first_level_results/
#       first_level_{SUBJECT}_Destrieux_{ses}_betas.npz
# Windows: primary=0.10–0.50s, early=0.10–0.30s, late=0.30–0.50s
# ROI groups: vmPFC=avg(vmPFC-lh, vmPFC-rh), dlPFC=avg(dlPFC-lh, dlPFC-rh)

import numpy as np

_WINDOWS = {
    "primary": (0.10, 0.50),
    "early":   (0.10, 0.30),
    "late":    (0.30, 0.50),
}
_ROI_GROUPS = {
    "vmPFC": ["vmPFC-lh", "vmPFC-rh"],
    "dlPFC": ["dlPFC-lh", "dlPFC-rh"],
}

for ses, betas_ses, times_ses in [("C", betas_C, times_C), ("S", betas_S, times_S)]:
    window_betas_by_group = {}
    full_betas_by_group   = {}

    for grp, members in _ROI_GROUPS.items():
        idx_members = [roi_list.index(m) for m in members if m in roi_list]
        if not idx_members:
            print(f"[WARN] {grp}: none of {members} found in roi_list, skipping")
            continue

        # full timecourse: avg across hemispheres → shape (n_reg, n_times)
        full_mat = np.mean(betas_ses[idx_members], axis=0)
        full_betas_by_group[grp] = full_mat

        # windowed means → shape (n_reg,) per window
        window_betas_by_group[grp] = {}
        for wname, (t_lo, t_hi) in _WINDOWS.items():
            mask_w = (times_ses >= t_lo) & (times_ses <= t_hi)
            if mask_w.sum() == 0:
                print(f"[WARN] {grp}/{ses}/{wname}: no timepoints in [{t_lo}, {t_hi}]")
                window_betas_by_group[grp][wname] = np.zeros(len(regressor_names))
            else:
                window_betas_by_group[grp][wname] = np.mean(full_mat[:, mask_w], axis=1)

    out_dir_ses = MEG_ROOT / SUBJECT / ses / "derivatives" / "fsaverage" / "first_level_results"
    out_dir_ses.mkdir(parents=True, exist_ok=True)
    out_path = out_dir_ses / f"first_level_{SUBJECT}_Destrieux_{ses}_betas.npz"

    np.savez_compressed(
        out_path,
        subject               = SUBJECT,
        session               = ses,
        regressor_names       = np.array(regressor_names),
        times                 = times_ses,
        window_betas_by_group = window_betas_by_group,
        full_betas_by_group   = full_betas_by_group,
        roi_groups            = list(_ROI_GROUPS.keys()),
        bandwidth             = 20,
        alpha_fdr             = 0.05,
    )
    print(f"[SAVED] {out_path}")
    for grp in _ROI_GROUPS:
        if grp in window_betas_by_group:
            for wn, arr in window_betas_by_group[grp].items():
                best_i = int(np.argmax(np.abs(arr)))
                print(f"  {grp}/{ses}/{wn}: max|β|={np.max(np.abs(arr)):.3e} @ {regressor_names[best_i]}")

print("\nCompatible NPZ export — ready for Group Analysis")


In [ ]:

# FULL SUMMARY TABLE: MODEL FITS + GLM (OLS + HAC-20)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

# 1. BEHAVIOURAL MODEL FITS
print("=" * 70)
print("1. BEHAVIOURAL MODEL PARAMETERS (fit_df)")
print("=" * 70)
if "fit_df" in dir() or "fit_df" in globals():
    _fit = fit_df.copy()
    key_cols = [c for c in [
        "block", "model", "fit_method", "fit_tau_mode", "n",
        "tau", "tau_env", "eta", "theta", "alpha_q", "beta", "eps",
        "alpha",
        "ll", "ll_block", "aic", "bic",
        "k_bias",
        "probe_active_mean", "p_lam_actor_gt_0.5",
        "n_confirm", "n_reject", "n_switch_in",
        "degenerate_probe",
    ] if c in _fit.columns]
    display(_fit[key_cols].round(5))
else:
    print("[!] fit_df not found")

# 2. LOG-LIKELIHOODS
print()
print("=" * 70)
print("2. LOG-LIKELIHOODS SUMMARY")
print("=" * 70)
_ll_rows = []
for _name, _val in [
    ("RL  C",          globals().get("rl_ll_c")),
    ("RL  S",          globals().get("rl_ll_s")),
    ("RL  combined",   globals().get("rl_ll")),
    ("PROBE C",        globals().get("ll_c")),
    ("PROBE S",        globals().get("ll_s")),
    ("PROBE combined", globals().get("ll_combined")),
]:
    if _val is not None:
        _ll_rows.append({"model_block": _name, "log-likelihood": float(_val)})
if _ll_rows:
    display(pd.DataFrame(_ll_rows).set_index("model_block"))
else:
    print("[!] ll variables not found")

# helper: collect rows from any glm_dict
def _glm_to_df(glm_dict, tval_key="tvals", reject_key="reject", pval_key="pvals"):
    rows = []
    for roi_name, blocks_dict in glm_dict.items():
        for block, res in blocks_dict.items():
            betas  = res["betas"]
            tvals  = res[tval_key]
            pvals  = res[pval_key]
            reject = res[reject_key]
            dof    = res.get("dof", np.nan)
            bw     = res.get("bandwidth", np.nan)
            times_b = times_C if block == "C" else times_S
            n_times = tvals.shape[1]
            for reg_idx, reg_name in enumerate(_reg_names[block]):
                t_row = tvals[reg_idx, :]
                p_row = pvals[reg_idx, :]
                r_row = reject[reg_idx, :]
                b_row = betas[reg_idx, :]
                valid = ~np.isnan(t_row)
                n_valid_tp = valid.sum()
                pct_fdr    = 100.0 * r_row.sum() / n_times if n_times > 0 else np.nan
                pct_raw    = 100.0 * (p_row < 0.05).sum() / n_times if n_times > 0 else np.nan
                mean_abs_t = float(np.nanmean(np.abs(t_row)))
                if n_valid_tp > 0:
                    peak_idx  = np.nanargmax(np.abs(t_row))
                    peak_t    = float(t_row[peak_idx])
                    peak_time = float(times_b[peak_idx])
                    peak_beta = float(b_row[peak_idx])   # NO rounding — values may be tiny
                    peak_p    = float(p_row[peak_idx])
                else:
                    peak_t = peak_time = peak_beta = peak_p = np.nan
                sig_idx     = np.where(r_row)[0]
                first_sig_t = float(times_b[sig_idx[0]])  if len(sig_idx) > 0 else np.nan
                last_sig_t  = float(times_b[sig_idx[-1]]) if len(sig_idx) > 0 else np.nan
                # mean beta over FDR-significant timepoints (NO rounding)
                sig_mask      = r_row.astype(bool)
                mean_beta_sig = float(np.nanmean(b_row[sig_mask])) if sig_mask.any() else np.nan
                rows.append({
                    "ROI": roi_name, "block": block, "regressor": reg_name,
                    "n_timepoints": n_times, "dof": dof, "bandwidth": bw,
                    "% sig FDR":     round(pct_fdr, 1),
                    "% sig raw":     round(pct_raw, 1),
                    "n sig FDR":     int(r_row.sum()),
                    "mean |t|":      round(mean_abs_t, 3),
                    "peak t":        round(peak_t, 3)       if not np.isnan(peak_t)    else np.nan,
                    "peak |t|":      round(abs(peak_t), 3)  if not np.isnan(peak_t)    else np.nan,
                    "peak time (s)": round(peak_time, 4)    if not np.isnan(peak_time) else np.nan,
                    "peak beta":     peak_beta,              # full float64 precision
                    "mean beta sig": mean_beta_sig,          # full float64 precision
                    "peak p":        round(peak_p, 5)        if not np.isnan(peak_p)    else np.nan,
                    "first sig (s)": round(first_sig_t, 4)  if not np.isnan(first_sig_t) else np.nan,
                    "last sig (s)":  round(last_sig_t, 4)   if not np.isnan(last_sig_t)  else np.nan,
                })
    return pd.DataFrame(rows)

# 3. GLM OLS - full table
print()
print("=" * 70)
print("3. GLM OLS (no autocorrelation correction)")
print("=" * 70)
df_glm_summary = _glm_to_df(glm_results, "tvals", "reject", "pvals")
display(df_glm_summary)

# 4. GLM HAC-20 - full table
print()
print("=" * 70)
print("4. GLM HAC-20 (Newey-West SE, bandwidth=20) — PRIMARY RESULT")
print("   Corrects inflated t at high regressor autocorrelation (ACF lag1 ≈ 0.83)")
print("=" * 70)
df_glm_hac = _glm_to_df(glm_hac, "tvals_hac", "reject_hac", "pvals_hac")
display(df_glm_hac)

# 4b. BETAS SUMMARY (HAC-20)
print()
print("=" * 70)
print("4b. BETAS (HAC-20) — peak β  and  mean β over FDR-sig timepoints")
print("    peak β  = β at timepoint with max |t-statistic|")
print("    mean β  = mean β over all HAC-20 FDR-significant timepoints (NaN = not sig)")
print("=" * 70)
_beta_fmt = lambda v: f"{v:.3e}" if pd.notna(v) and v != 0 else ("NaN" if pd.isna(v) else "0")
for blk in ["C", "S"]:
    _sub_b = df_glm_hac[df_glm_hac["block"] == blk]
    print(f"\n  [Block {blk}]  Peak β  (scientific notation):")
    _piv = _sub_b.pivot(index="regressor", columns="ROI", values="peak beta")
    display(_piv.map(_beta_fmt))
    print(f"\n  [Block {blk}]  Mean β  (FDR-sig timepoints):")
    _piv2 = _sub_b.pivot(index="regressor", columns="ROI", values="mean beta sig")
    display(_piv2.map(_beta_fmt))

# 5. PIVOT TABLES: OLS vs HAC-20  (% FDR sig)
print()
print("=" * 70)
print("5. PIVOT % sig FDR:  OLS  vs  HAC-20")
print("=" * 70)

for blk in ["C", "S"]:
    sub_ols = df_glm_summary[df_glm_summary["block"] == blk].pivot(
        index="regressor", columns="ROI", values="% sig FDR").round(1)
    sub_hac = df_glm_hac[df_glm_hac["block"] == blk].pivot(
        index="regressor", columns="ROI", values="% sig FDR").round(1)
    combined = pd.concat({"OLS": sub_ols, "HAC-20": sub_hac}, axis=1)
    combined.columns = [f"{m} | {r}" for m, r in combined.columns]
    print(f"\n  Block {blk}:")
    display(combined)

# 6. VISUALISATION: heatmap % FDR sig — OLS vs HAC-20 (all regressors × ROI)
_roi_order = sorted(target_roi_indices.keys())
_reg_order = regressor_names

fig, axes_grid = plt.subplots(2, 2, figsize=(16, 9),
                               gridspec_kw={"wspace": 0.35, "hspace": 0.4})
fig.suptitle(f"% FDR-significant timepoints  |  subject: {SUBJECT}\n"
             f"Left: OLS (standard SE)  |  Right: HAC-20 (Newey-West, bw=20)",
             fontsize=11)

_titles = ["[C] OLS", "[C] HAC-20", "[S] OLS", "[S] HAC-20"]
_srcs   = [
    (df_glm_summary, "C"), (df_glm_hac, "C"),
    (df_glm_summary, "S"), (df_glm_hac, "S"),
]
for ax_flat, (title, (df_src, blk)) in zip(axes_grid.flat, zip(_titles, _srcs)):
    mat = np.zeros((len(_reg_order), len(_roi_order)))
    sub = df_src[df_src["block"] == blk]
    for ri, reg in enumerate(_reg_order):
        for ci, roi in enumerate(_roi_order):
            row = sub[(sub["regressor"] == reg) & (sub["ROI"] == roi)]
            if len(row):
                mat[ri, ci] = row["% sig FDR"].values[0]
    im_h = ax_flat.imshow(mat, vmin=0, vmax=100, cmap="YlOrRd", aspect="auto")
    ax_flat.set_xticks(range(len(_roi_order)))
    ax_flat.set_xticklabels([r.replace("superiorfrontal", "supfr")
                              .replace("-lh", "\n-lh").replace("-rh", "\n-rh")
                              for r in _roi_order], fontsize=7)
    ax_flat.set_yticks(range(len(_reg_order)))
    ax_flat.set_yticklabels(_reg_order, fontsize=7)
    ax_flat.set_title(title, fontsize=9, fontweight="bold")
    for ri in range(len(_reg_order)):
        for ci in range(len(_roi_order)):
            v = mat[ri, ci]
            ax_flat.text(ci, ri, f"{v:.0f}", ha="center", va="center",
                         fontsize=6, color="white" if v > 55 else "black")
    plt.colorbar(im_h, ax=ax_flat, fraction=0.03, pad=0.02).set_label("%", fontsize=7)

plt.tight_layout()
_fig_path = MODEL_OUT / "glm_heatmap_ols_vs_hac20.png"
fig.savefig(_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"[SAVED] Heatmap -> {_fig_path}")

# 7. AUTOCORRELATION DIAGNOSTICS
print()
print("=" * 70)
print("6. AUTOCORRELATION DIAGNOSTICS")
print("=" * 70)

_lam0_col = design_C[:, regressor_names.index("lambda0_post")]
_acf1_C   = np.corrcoef(_lam0_col[:-1], _lam0_col[1:])[0, 1]
_lam0_col_S = design_S[:, regressor_names.index("lambda0_post")]
_acf1_S   = np.corrcoef(_lam0_col_S[:-1], _lam0_col_S[1:])[0, 1]
_eff_n_C  = ok_C.sum() * (1 - _acf1_C) / (1 + _acf1_C)
_eff_n_S  = ok_S.sum() * (1 - _acf1_S) / (1 + _acf1_S)

print(f"  lambda0_post ACF(lag=1):  C={_acf1_C:.3f}  S={_acf1_S:.3f}")
print(f"  Actual trials:            C={ok_C.sum()}  S={ok_S.sum()}")
print(f"  Effective (AR(1)):        C≈{_eff_n_C:.0f}  S≈{_eff_n_S:.0f}")
print(f"  OLS SE underestimated ~{np.sqrt(ok_C.sum()/_eff_n_C):.1f}x  "
      f"→ t inflated ~{np.sqrt(ok_C.sum()/_eff_n_C):.1f}x")
print(f"  HAC bandwidth=20 corrects first 20 autocovariance lags")
print()

_lam_idx = regressor_names.index("lambda0_post")
_survives = []
for roi in target_roi_indices:
    for blk in ["C", "S"]:
        p20 = df_glm_hac[(df_glm_hac["ROI"] == roi) &
                         (df_glm_hac["block"] == blk) &
                         (df_glm_hac["regressor"] == "lambda0_post")]["% sig FDR"].values
        _survives.append(float(p20[0]) > 5 if len(p20) else False)

_n_surv = sum(_survives)
_n_total = len(_survives)
print(f"  lambda0_post at HAC-20: {_n_surv}/{_n_total} ROI×block have >5% FDR-sig timepoints")
if _n_surv > _n_total // 2:
    print("  → VERDICT: real neural effect (survives conservative correction)")
else:
    print("  → VERDICT: effect gone at HAC-20 — likely autocorrelation artifact")

# 8. EXPORT ALL CSVs
print()
print("=" * 70)
print("7. SAVING FILES")
print("=" * 70)

_out_ols   = MODEL_OUT / "glm_summary_ols.csv"
_out_hac   = MODEL_OUT / "glm_summary_hac20.csv"
_out_betas = MODEL_OUT / "glm_betas_summary.csv"
_out_fit   = MODEL_OUT / "fit_summary.csv"
_out_notes = MODEL_OUT / "glm_analysis_notes.txt"

df_glm_summary.to_csv(_out_ols, index=False)
df_glm_hac.to_csv(    _out_hac, index=False)

_beta_cols = ["ROI", "block", "regressor",
              "% sig FDR", "peak t", "peak time (s)",
              "peak beta", "mean beta sig",
              "first sig (s)", "last sig (s)"]
df_glm_hac[[c for c in _beta_cols if c in df_glm_hac.columns]].to_csv(_out_betas, index=False)

if "fit_df" in globals():
    fit_df.to_csv(_out_fit, index=False)
    print(f"  [SAVED] Model fit      -> {_out_fit}")

# Build beta lines for notes (only significant rows)
def _fmt_b(v):
    return f"{v:.3e}" if pd.notna(v) and v != 0 else ("NaN" if pd.isna(v) else "0")

_beta_note_lines = []
for blk in ["C", "S"]:
    _sig_rows = df_glm_hac[(df_glm_hac["block"] == blk) & (df_glm_hac["% sig FDR"] > 5)]
    for _, brow in _sig_rows.iterrows():
        _beta_note_lines.append(
            f"  {brow['ROI']}/{blk}  {brow['regressor']:22s}"
            f"  peak_β={_fmt_b(brow['peak beta'])}  mean_β_sig={_fmt_b(brow['mean beta sig'])}"
            f"  peak_t={brow['peak t']:+.2f} @{brow['peak time (s)']:.3f}s"
            f"  FDR={brow['% sig FDR']:.0f}%"
        )

_notes = f"""GLM ANALYSIS NOTES — {SUBJECT}
Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}
============================================================

DATA
  Subject:      {SUBJECT}
  Epoch lock:   {EPOCH_LOCK}
  Trials C:     {ok_C.sum()} (of {epochs_C.__len__()})
  Trials S:     {ok_S.sum()} (of {epochs_S.__len__()})
  ROI:          {list(target_roi_indices.keys())}
  Regressors:   {regressor_names}

BEHAVIOUR MODEL
  Fit type:     {FIT_TAU_MODE}
  PROBE > RL:   C={ll_c > rl_ll_c} (ΔLL={ll_c - rl_ll_c:.1f}), S={ll_s > rl_ll_s} (ΔLL={ll_s - rl_ll_s:.1f})

GLM
  Method OLS:   standard SE, FDR BH α=0.05
  Method HAC-20: Newey-West SE bandwidth=20, FDR BH α=0.05
  Regressor orthogonalisation: {ORTHOGONALIZE_SEQUENCE}

AUTOCORRELATION DIAGNOSTICS
  lambda0_post ACF(lag=1): C={_acf1_C:.3f}, S={_acf1_S:.3f}
  Effective N: C≈{_eff_n_C:.0f}, S≈{_eff_n_S:.0f}
  OLS SE underestimated ≈{np.sqrt(ok_C.sum()/_eff_n_C):.1f}x → using HAC-20

BETAS (HAC-20, significant rows only, >5% FDR)
  Format: peak_β = β at max |t|  |  mean_β_sig = mean β over FDR-sig timepoints
{chr(10).join(_beta_note_lines) if _beta_note_lines else "  (none significant)"}

OUTPUT FILES
  glm_summary_ols.csv    — full table (OLS)
  glm_summary_hac20.csv  — full table (HAC-20, PRIMARY)
  glm_betas_summary.csv  — beta coefficients (HAC-20, key columns)
  glm_heatmap_ols_vs_hac20.png — comparison heatmap
  fit_summary.csv        — behavioural model parameters
"""
with open(_out_notes, "w", encoding="utf-8") as _f:
    _f.write(_notes)

print(f"  [SAVED] GLM OLS        -> {_out_ols}")
print(f"  [SAVED] GLM HAC-20     -> {_out_hac}")
print(f"  [SAVED] GLM Betas      -> {_out_betas}")
print(f"  [SAVED] Notes          -> {_out_notes}")

print()
print("Available variables:")
print("  df_glm_summary  — OLS results")
print("  df_glm_hac      — HAC-20 results (primary; betas at full float64 precision)")
piv_C = df_glm_hac[df_glm_hac["block"] == "C"].pivot(
    index="regressor", columns="ROI", values="% sig FDR").round(1)
piv_S = df_glm_hac[df_glm_hac["block"] == "S"].pivot(
    index="regressor", columns="ROI", values="% sig FDR").round(1)
print("  piv_C, piv_S    — % FDR sig HAC-20 (updated)")


In [ ]:

# BAR PLOT: dlPFC vs vmPFC — peak β и mean β (FDR-sig) per regressor
# HAC-20 Newey-West, bw=20

import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np
import pandas as pd
import math

# Pick scale so that a typical significant beta displays as ~10 units
_pb_sig = df_glm_hac[df_glm_hac["% sig FDR"] > 5]["peak beta"].abs().dropna()
if len(_pb_sig) and _pb_sig.max() > 0:
    _ref_val = _pb_sig.quantile(0.75)
    _mag     = math.floor(math.log10(abs(_ref_val)))   # e.g. -5 for 1e-5
    _SCALE   = 10 ** (-_mag + 1)   # → typical sig beta displays as ~10
    _SCALE_L = f"×10^{_mag}"       # axis label: "β ×10^-5" means raw β was 1e-5
else:
    _SCALE, _SCALE_L = 1.0, "(raw)"

print(f"[INFO] auto scale ×{_SCALE:.0e}  "
      f"(p75 sig |peak β| = {_pb_sig.quantile(0.75) if len(_pb_sig) else 0:.2e}  "
      f"→ display ≈ {_pb_sig.quantile(0.75)*_SCALE if len(_pb_sig) else 0:.1f})")

_roi_groups = {"dlPFC": ["dlPFC-lh", "dlPFC-rh"],
               "vmPFC": ["vmPFC-lh", "vmPFC-rh"]}
_grp_colors = {"dlPFC": "#1565C0", "vmPFC": "#C62828"}   # dark blue / dark red

n_reg = len(regressor_names)
x     = np.arange(n_reg)
width = 0.35

def _grp_vals(sub_df, members, metric):
    means, sigs, lh_list, rh_list = [], [], [], []
    for reg in regressor_names:
        pb_list, sig_any = [], False
        for roi_m in members:
            r_ = sub_df[(sub_df["regressor"] == reg) & (sub_df["ROI"] == roi_m)]
            if len(r_):
                pb = r_[metric].values[0]
                pb_list.append((pb * _SCALE) if pd.notna(pb) else 0.0)
                if r_["% sig FDR"].values[0] > 5:
                    sig_any = True
            else:
                pb_list.append(0.0)
        means.append(np.mean(pb_list))
        sigs.append(sig_any)
        lh_list.append(pb_list[0])
        rh_list.append(pb_list[1] if len(pb_list) > 1 else pb_list[0])
    stds = [abs(lh - rh) / 2 for lh, rh in zip(lh_list, rh_list)]
    return means, stds, sigs, lh_list, rh_list

# plotting function
def _make_barplot(metric, metric_label, fig_suffix):
    fig_bp, axes_bp = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    fig_bp.suptitle(
        f"dlPFC vs vmPFC — {metric_label} ({_SCALE_L}) per regressor  (HAC-20, bw=20)\n"
        f"{SUBJECT}  |  ★ = ≥1 hemisphere >5% FDR-sig  |  ●=lh  ■=rh  |  bar=avg(lh+rh)",
        fontsize=10)

    for ax_bp, blk in zip(axes_bp, ["C", "S"]):
        sub_bp   = df_glm_hac[df_glm_hac["block"] == blk]
        star_q   = []    # (x_pos, y_val, std, color)
        all_vals = []    # for ylim

        for gi, (grp, members) in enumerate(_roi_groups.items()):
            offset = (gi - 0.5) * width
            means, stds, sigs, lh_vals, rh_vals = _grp_vals(sub_bp, members, metric)
            all_vals.extend(lh_vals + rh_vals)

            ax_bp.bar(x + offset, means, width,
                      color=_grp_colors[grp], alpha=0.75,
                      edgecolor="white", linewidth=0.5)
            ax_bp.errorbar(x + offset, means, yerr=stds,
                           fmt="none", color="black",
                           capsize=3, linewidth=0.8, alpha=0.4)
            ax_bp.scatter(x + offset - 0.07, lh_vals, s=26, marker="o", zorder=5,
                          color=_grp_colors[grp], edgecolor="white",
                          linewidth=0.4, alpha=0.95)
            ax_bp.scatter(x + offset + 0.07, rh_vals, s=26, marker="s", zorder=5,
                          color=_grp_colors[grp], edgecolor="white",
                          linewidth=0.4, alpha=0.6)

            for xi, (val, sd, is_sig) in enumerate(zip(means, stds, sigs)):
                if is_sig:
                    star_q.append((xi + offset, val, sd, _grp_colors[grp]))

        # tight y-axis around actual data range
        nonzero_vals = [v for v in all_vals if abs(v) > 1e-15]
        if nonzero_vals:
            _max_abs = max(abs(v) for v in nonzero_vals)
            _ylim    = _max_abs * 1.45
        else:
            _ylim = 1.0
        ax_bp.set_ylim(-_ylim, _ylim)

        # draw stars after ylim is set
        ylo, yhi = ax_bp.get_ylim()
        y_off = (yhi - ylo) * 0.04
        for (xp, yv, sd, col) in star_q:
            ys = yv + sd + y_off if yv >= 0 else yv - sd - y_off
            ax_bp.text(xp, ys, "★", ha="center",
                       va="bottom" if yv >= 0 else "top",
                       fontsize=8, color=col, fontweight="bold")

        ax_bp.axhline(0, color="black", linewidth=0.7, linestyle="--", alpha=0.35)
        ax_bp.set_title(f"Block {blk}", fontsize=10, fontweight="bold")
        ax_bp.set_ylabel(f"{metric_label} ({_SCALE_L})", fontsize=9)
        ax_bp.grid(axis="y", alpha=0.22, linewidth=0.5)

    axes_bp[-1].set_xticks(x)
    axes_bp[-1].set_xticklabels(regressor_names, rotation=40, ha="right", fontsize=8)

    _handles = [
        mlines.Line2D([], [], color=_grp_colors["dlPFC"], linewidth=7,
                      solid_capstyle="butt", label="dlPFC (avg lh+rh)"),
        mlines.Line2D([], [], color=_grp_colors["vmPFC"], linewidth=7,
                      solid_capstyle="butt", label="vmPFC (avg lh+rh)"),
        mlines.Line2D([], [], marker="o", linestyle="none", color="gray",
                      markersize=6, label="lh hemisphere"),
        mlines.Line2D([], [], marker="s", linestyle="none", color="gray",
                      markersize=6, label="rh hemisphere"),
        mlines.Line2D([], [], marker="*", linestyle="none", color="gray",
                      markersize=9, label="★ FDR >5%"),
    ]
    axes_bp[0].legend(handles=_handles, fontsize=8, loc="upper right")

    fig_bp.tight_layout()
    _fp = MODEL_OUT / f"glm_barplot_dlpfc_vmpfc_{fig_suffix}.png"
    fig_bp.savefig(_fp, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[SAVED] {_fp}")

# generate both plots
_make_barplot("peak beta",     "Peak β",              "peakbeta")
_make_barplot("mean beta sig", "Mean β (FDR-sig tp)", "meanbeta")
